# 31 Benchmark Runtime dan Failure Audit


In [6]:
from pathlib import Path
import os, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
ROOT_31 = Path(os.environ.get('RUNTIME31_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_31 / 'LIB').is_dir():
    raise FileNotFoundError('Folder LIB belum tersedia: ' + str(ROOT_31))
print('Proyek:', ROOT_31)


Proyek: /content/drive/MyDrive/REVISED


## Ruang lingkup dan reuse

| Bagian | Cakupan | Perhitungan |
|---|---|---|
| Kualitas dan statistik | 8 metode; 10 benchmark | Reuse tahap 25 setelah validasi |
| Audit historis benchmark | 440 full + 2.400 resample | Audit angka dan sumber, tanpa fitting |
| Runtime baru | CARE-KM dan seluruh tujuh baseline | 10 full dataset × 8 metode × 30 blok = 2.400 attempts |
| Warm-up awal | 8 metode × 10 dataset | 80 run, terpisah dari denominator timing |

Semua 3.272 catatan tahap 25 dan 55 verifikasi initializer diperiksa.
Sebanyak 310 run BK-Modes benchmark sudah ada; tidak mengulang eksperimen
kualitas itu. Matriks, loader, diskretisasi, urutan fitur/baris dan hash
harus sesuai sumber. Nilai k mengikuti registry benchmark (2–5), bukan
k=16 LMS. CARE-KM tetap q=16, separation, min_gain=0 dan batas refinement 4.
Optimizer akhir, tie handling dan max_iter=100 tidak diubah.

Runtime memerlukan eksekusi initializer dan optimizer baru. Waktu lama yang
ditandai tidak comparable tidak digunakan untuk klaim biaya komputasi baru.
Skor kualitas/ranks/Friedman/Wilcoxon tetap dari sumber delapan metode.
Tidak ada tuning, pemilihan best run, atau penggantian implementasi.


## Delapan metode dan seed yang dipertahankan

- Deterministik: Cao, Bai, CARE-KM, BK-Modes. Seed API tetap 120000.
- Stokastik pada implementasi ini: Random, Huang, K-Modes++, HR-2023.
  Pakai sepuluh seed full-benchmark tahap 25: **120000–120009**.

Blok 0–9 memakai sepuluh seed tersebut; blok 10–19 dan 20–29 mengulang
urutan yang sama. Jadi setiap metode stokastik memiliki **10 seed × 3
pengukuran = 30 attempts**, bukan memilih seed terbaik. Semua delapan metode
memakai matriks, hardware, batas thread, definisi waktu dan jumlah pengukuran
yang sama. Keempat stokastik memperoleh seed yang sama dalam blok yang sama.

Cost/iterasi diperiksa terhadap **run sumber dengan seed yang sama**, bukan
terhadap mean atau run 0 untuk semua seed. Fingerprint konsistensi stokastik
dibandingkan dalam seed yang sama; perbedaan antar-seed merupakan variasi
yang sah dan tidak disamarkan menjadi failure. Ada ringkasan pooled 30
pengukuran serta ringkasan per seed. Pengukuran ini tidak mengubah skor
kualitas tahap 25 atau memilih ulang konfigurasi.

Versi ini membentuk campaign baru berisi delapan metode. Timing dari versi
empat metode tetap tersimpan sebagai arsip, dan tidak dicampur ke tabel baru.


In [7]:
KONFIGURASI_31 = {
    'benchmark_datasets': None,  # None = seluruh 10 dataset pada registry
    'timing_blocks': 30,
    'warmup_repetitions': 1,
    'force_timing': False,
    'force_audit': False,
}


## Masukan DATA yang diperiksa

```
DATA/dataset_car+evaluation.zip
DATA/dataset_lymphography.zip
DATA/hayes+roth.zip
DATA/statlog+australian+credit+approval.zip
DATA/chronic+kidney+disease.zip
DATA/annealing.zip
DATA/statlog+heart.zip
DATA/heart+disease.zip
DATA/thyroid+disease.zip
DATA/dataset_nursery.zip
```

Jika sudah tersedia di Drive, tidak perlu upload ulang. Nama dan SHA256 harus
sesuai manifest tahap 25. Jika belum lengkap, hasil kualitas/audit tetap
disimpan dan timing dihentikan dengan daftar yang perlu dilengkapi pada
`PROTOKOL/04_pemeriksaan_DATA.csv`. Tidak ada pencarian dataset pengganti.

Chronic Kidney Disease dapat memakai fallback ARFF yang didukung loader lama;
loader yang sama akan digunakan. Jika arsip CKD memerlukan RAR extractor,
loader proyek menyediakan penanganannya pada Colab. Matriks akhirnya tetap
wajib sama dengan fingerprint sumber sebelum timing diterima.


## Definisi waktu dan kebijakan cache

- `init_sec`: seluruh pemanggilan initializer. BK-Modes mencakup seleksi
  leaf, seluruh Two-Modes/biseksi dan pembentukan mode. Semua biaya parent
  dan refinement CARE-KM juga termasuk.
- `kmodes_sec`: optimizer K-Modes akhir dari mode awal yang baru dihitung.
- `total_sec = init_sec + kmodes_sec`: end-to-end dua tahap clustering ini.

Data disiapkan di luar timer; DBI/SI/ARI, validasi, hashing, penulisan Drive
dan plotting berada di luar timer. Ukur dalam satu sesi CPU, thread dibatasi
sama, dan urutan delapan metode dirotasi antarblok. Catat CPU, RAM, versi,
threadpool serta identitas boot VM. Ringkasan melaporkan median/IQR dan
mean/SD, dengan failure denominator tetap terlihat.

Cache per attempt memeriksa isi, kode, konfigurasi dan mesin. Pengukuran
selesai dapat dilewati setelah interupsi pada VM yang sama; VM berbeda
memicu campaign timing baru. `force_timing=True` mengukur ulang pada
konfigurasi sama; rekaman pengukuran sebelumnya tetap diarsipkan, termasuk
kegagalannya. `force_audit=True` menyusun ulang keluaran audit/tabel
historis tanpa mengulang fitting sumber. Perubahan output tampilan notebook
tidak membatalkan cache. Pengaturan kurang dari 30 blok atau sebagian dataset
ditandai diagnostik (`full_protocol=False`). Jumlah blok yang bukan kelipatan
10 juga diagnostik agar cakupan pengulangan sepuluh seed tidak disalahartikan.

Kompatibilitas cache autosave diperiksa sebelum pemakaian ulang. Cache dari versi delapan metode sebelumnya hanya
diadopsi jika semua fungsi ilmiah identik, sumber/config/runtime/VM sama,
serta checksum catatan dan isi berkas keluarannya valid. Catatan asal tetap
tersimpan; pemindahan referensi dicatat di `KONTROL/CAMPAGNES/.../COMPATIBILITY_MIGRATION/`.
Cache empat metode dan pengukuran dari VM berbeda tetap tidak digunakan.


In [8]:
UTILS31_SOURCE = "from pathlib import Path\nimport ast,gzip,hashlib,importlib.metadata,io,json,platform,sys,uuid\nimport numpy as np\nimport pandas as pd\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef digest(value):\n    return sha(json.dumps(clean(value), sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8'))\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {str(k): clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [clean(v) for v in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, np.bool_):\n        return bool(value)\n    return value\n\ndef atomic(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + '.' + uuid.uuid4().hex[:8] + '.tmp')\n    temp.write_bytes(data)\n    temp.replace(path)\n\ndef save_json(path, value):\n    atomic(path, json.dumps(clean(value), ensure_ascii=False, indent=2, allow_nan=False).encode('utf-8'))\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False, lineterminator='\\n', float_format='%.17g').encode('utf-8')\n    atomic(path, gzip.compress(data, mtime=0) if str(path).endswith('.gz') else data)\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding='utf-8'))\n        return obj if isinstance(obj, dict) else {}\n    except (OSError, ValueError, UnicodeError):\n        return {}\n\nclass Files:\n\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root):\n            raise ValueError('Lokasi di luar proyek: ' + relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative)\n        notebook = path.suffix.lower() == '.ipynb'\n        # Colab may save cell outputs while the same notebook is being read.\n        # Wait for a complete, stable snapshot; never ignore code differences.\n        for attempt in range(10 if notebook else 1):\n            try:\n                before = path.stat()\n                stamp = (before.st_size, before.st_mtime_ns, before.st_ctime_ns)\n                if relative in self.cache and self.cache[relative][0] == stamp:\n                    return self.cache[relative][1]\n                data = path.read_bytes()\n                after = path.stat()\n                if stamp != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):\n                    raise RuntimeError('Berkas berubah ketika dibaca: ' + relative)\n                if notebook:\n                    nb = json.loads(data.decode('utf-8-sig'))\n                    if not isinstance(nb, dict) or not isinstance(nb.get('cells'), list):\n                        raise ValueError('Struktur notebook belum lengkap: ' + relative)\n                self.cache[relative] = (stamp, data)\n                return data\n            except (OSError, RuntimeError, UnicodeError, ValueError):\n                if not notebook or attempt == 9:\n                    raise\n                import time\n                time.sleep(min(.1 * 2 ** attempt, .8))\n\n    def hash(self, relative):\n        return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith('.gz'):\n            data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data), dtype=str if strings else None, keep_default_na=not strings, float_precision='round_trip')\n\ndef runtime():\n    packages = {}\n    for name in ['numpy', 'pandas', 'scipy', 'sklearn', 'matplotlib']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod, '__version__', None) or importlib.metadata.version('scikit-learn' if name == 'sklearn' else name)\n    return {'python': platform.python_version(), 'packages': packages}\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read('PYTHON/' + stem + '.ipynb').decode('utf-8-sig'))\n    code = []\n    for cell in nb['cells']:\n        if cell['cell_type'] != 'code':\n            continue\n        source = ''.join(cell['source'])\n        if tag in cell.get('metadata', {}).get('tags', []):\n            tree = ast.parse(source)\n            if len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assign) or len(tree.body[0].targets) != 1 or (not isinstance(tree.body[0].targets[0], ast.Name)) or (tree.body[0].targets[0].id != variable) or (not isinstance(ast.literal_eval(tree.body[0].value), dict)):\n                raise RuntimeError('Sel konfigurasi hanya boleh berisi literal ' + variable)\n        else:\n            code.append(source)\n    return digest(code)\n\ndef cache_read(files, path, signature, force=False):\n    record = read_checkpoint(path)\n    checksum = record.pop('record_sha256', '')\n    try:\n        valid = digest(record) == checksum\n    except (TypeError, ValueError):\n        valid = False\n    if force or not valid or record.get('signature') != signature or (record.get('status') != 'selesai'):\n        return None\n    if not all((files.path(p).is_file() and files.hash(p) == h for p, h in record.get('outputs', {}).items())):\n        return None\n    return record\n\ndef cache_write(files, path, signature, outputs, extra):\n    record = {'status': 'selesai', 'signature': signature, 'outputs': {str(p.relative_to(files.root)).replace('\\\\', '/'): sha(p.read_bytes()) for p in outputs}, **extra}\n    record['record_sha256'] = digest(record)\n    save_json(path, record)\n    return record\n\ndef unchanged(files, watched, code_checks):\n    for path, checksum in watched.items():\n        if files.hash(path) != checksum:\n            raise RuntimeError('Masukan berubah selama eksekusi: ' + path)\n    for stem, tag, var, expected in code_checks:\n        if notebook_hash(files, stem, tag, var) != expected:\n            raise RuntimeError('Kode notebook berubah selama eksekusi: ' + stem)\n\nFiles.sha = Files.hash\n"

_u31 = types.ModuleType('_utils31')
exec(compile(UTILS31_SOURCE, '_utils31', 'exec'), _u31.__dict__)

HELPER25_SOURCE = '"""Eight-initializer experiments, shared optimizer, persistent per-run artifacts."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport hashlib\nimport importlib.metadata\nimport io\nimport itertools\nimport json\nimport os\nimport platform\nimport re\nimport sys\nimport time\nimport types\nimport uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE = \'25_Perbandingan_Initializer_Terintegrasi\'\nOLD_STAGE = \'25_Verifikasi_Baseline_BK_Modes_2025\'\nMETHODS = [\'Random\', \'Huang\', \'Cao\', \'Bai\', \'K-Modes++\', \'HR-2023\', \'CARE-KM\', \'BK-Modes\']\nSTOCHASTIC = {\'Random\', \'Huang\', \'K-Modes++\', \'HR-2023\'}\nSCOPES = [\'LMS_FULL\', \'LMS_RESAMPLING\', \'LMS_TRAIN\', \'PENDIKOM_TRAIN\',\n          \'BENCHMARK_FULL\', \'BENCHMARK_RESAMPLING\']\nMETRICS = [\'initial_cost\', \'final_cost\', \'dbi\', \'si\', \'iterations\']\nDEFAULT_CONFIG = {\'scopes\': SCOPES, \'lms_stochastic_runs\': 15, \'holdout_stochastic_runs\': 15,\n                  \'benchmark_stochastic_runs\': 10, \'resample_runs\': 30,\n                  \'save_semantic_partitions\': True, \'reuse_previous_metrics\': True,\n                  \'force\': False}\nINPUT_LMS = \'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv\'\nTRAIN_INPUTS = {\n    \'LMS_TRAIN\': (\'LMS\', \'user\', 4660, 1998),\n    \'PENDIKOM_TRAIN\': (\'PENDIKOM\', \'userid\', 322, 139),\n}\nOLD_SOURCES = {\n    \'LMS_FULL\': (\'07_Eksperimen_LMS_Full\', \'OUTPUT/EKSPERIMEN/LMS/07_Eksperimen_LMS_Full/04_full_dataset_raw.csv\', \'rep\'),\n    \'LMS_RESAMPLING\': (\'09_Robustness_LMS_30x80\', \'OUTPUT/EKSPERIMEN/LMS/09_Robustness_LMS_30x80/06_robustness_raw.csv\', \'rep\'),\n    \'PENDIKOM_TRAIN\': (\'22_Holdout_LMS_Pendikom\', \'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_clustering_baseline_raw.csv\', \'rep\'),\n    \'BENCHMARK_FULL\': (\'19_Generalisasi_Benchmark\', \'OUTPUT/EKSPERIMEN/GENERALISASI/19_Generalisasi_Benchmark/10_cross_dataset_full_raw.csv\', \'run\'),\n    \'BENCHMARK_RESAMPLING\': (\'19_Generalisasi_Benchmark\', \'OUTPUT/EKSPERIMEN/GENERALISASI/19_Generalisasi_Benchmark/10_cross_dataset_resample_raw.csv\', \'rep\'),\n}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')).hexdigest()\n\n\ndef binary_hash(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {str(k): clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [clean(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return clean(value.tolist())\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating, float)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, np.bool_):\n        return bool(value)\n    return value\n\n\ndef save_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(clean(value), indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef save_csv(path, frame):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    frame.to_csv(temp, index=False, lineterminator=\'\\n\')\n    temp.replace(path)\n\n\ndef read_checkpoint(path):\n    try:\n        data = json.loads(path.read_text(encoding=\'utf-8\'))\n        return data if isinstance(data, dict) else {}\n    except (OSError, UnicodeError, json.JSONDecodeError):\n        return {}\n\n\nclass Files:\n    """Fresh content hashes for each call; reuse reads only while stat is unchanged."""\n    def __init__(self, root):\n        self.root, self.cache = Path(root).resolve(), {}\n\n    def read(self, relative):\n        path = self.root / relative\n        before = path.stat()\n        stamp = (before.st_size, before.st_mtime_ns, before.st_ctime_ns)\n        if relative in self.cache and self.cache[relative][0] == stamp:\n            return self.cache[relative][1]\n        data = path.read_bytes()\n        after = path.stat()\n        if stamp != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):\n            raise RuntimeError(\'Berkas berubah saat dibaca: \' + relative)\n        self.cache[relative] = (stamp, data)\n        return data\n\n    def sha(self, relative):\n        return binary_hash(self.read(relative))\n\n\ndef runtime():\n    packages = {}\n    for name, module in [(\'numpy\', \'numpy\'), (\'pandas\', \'pandas\'), (\'scipy\', \'scipy\'), (\'sklearn\', \'sklearn\')]:\n        loaded = sys.modules.get(module)\n        packages[name] = getattr(loaded, \'__version__\', None) or importlib.metadata.version(\'scikit-learn\' if name == \'sklearn\' else name)\n    return {\'python\': platform.python_version(), \'packages\': packages}\n\n\ndef literals(source, requested):\n    result = {}\n    for node in ast.parse(source.decode(\'utf-8-sig\')).body:\n        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):\n            name = node.targets[0].id\n            if name in requested:\n                result[name] = ast.literal_eval(node.value)\n    if set(result) != set(requested):\n        raise RuntimeError(\'Konfigurasi pustaka belum lengkap: \' + str(set(requested) - set(result)))\n    return result\n\n\ndef load_module(source, name, filename):\n    module = types.ModuleType(name)\n    module.__file__ = str(filename)\n    exec(compile(source, str(filename), \'exec\'), module.__dict__)\n    return module\n\n\ndef load_project(files):\n    core = load_module(files.read(\'LIB/care_km.py\'), \'_initializer25_core\', files.root / \'LIB/care_km.py\')\n    gd = load_module(files.read(\'LIB/generalization_data.py\'), \'_initializer25_datasets\', files.root / \'LIB/generalization_data.py\')\n    constants = literals(files.read(\'LIB/experiment_lib.py\'), [\'K\', \'MAX_ITER\', \'SI_SAMPLE_SIZE\', \'FROZEN_CARE_CONFIG\'])\n    features = literals(files.read(\'LIB/pipeline_data.py\'), [\'MAIN_FEATURES\', \'PENDIKOM_FEATURES\'])\n    features[\'PENDIKOM_USED\'] = [c for c in features[\'PENDIKOM_FEATURES\'] if c != \'navigation_skip\']\n    if constants[\'K\'] != 16 or constants[\'MAX_ITER\'] != 100:\n        raise RuntimeError(\'Protokol utama harus k=16 dan MAX_ITER=100 untuk paket ini.\')\n    return core, gd, constants, features\n\n\ndef read_frame(files, path):\n    return pd.read_csv(io.BytesIO(files.read(path)), dtype=str, keep_default_na=False)\n\n\ndef validate_config(configuration, gd):\n    unknown = set(configuration) - set(DEFAULT_CONFIG) - {\'benchmark_datasets\'}\n    if unknown:\n        raise ValueError(\'Nama konfigurasi tidak dikenal: \' + str(sorted(unknown)))\n    cfg = dict(DEFAULT_CONFIG)\n    cfg.update(configuration)\n    cfg[\'scopes\'] = list(cfg[\'scopes\'])\n    cfg[\'benchmark_datasets\'] = list(cfg.get(\'benchmark_datasets\', gd.DATASET_ORDER))\n    if not cfg[\'scopes\'] or len(set(cfg[\'scopes\'])) != len(cfg[\'scopes\']) or set(cfg[\'scopes\']) - set(SCOPES):\n        raise ValueError(\'Pilihan kelompok eksperimen tidak sesuai daftar.\')\n    if not cfg[\'benchmark_datasets\'] or len(set(cfg[\'benchmark_datasets\'])) != len(cfg[\'benchmark_datasets\']) or set(cfg[\'benchmark_datasets\']) - set(gd.DATASET_ORDER):\n        raise ValueError(\'Pilihan benchmark tidak sesuai registry.\')\n    for key in [\'lms_stochastic_runs\', \'holdout_stochastic_runs\', \'benchmark_stochastic_runs\', \'resample_runs\']:\n        if isinstance(cfg[key], bool) or not isinstance(cfg[key], int) or cfg[key] < 1:\n            raise ValueError(\'Jumlah pengulangan tidak valid: \' + key)\n    if cfg[\'resample_runs\'] > 30:\n        raise ValueError(\'Paket menyediakan paling banyak 30 indeks resampling.\')\n    for key in [\'save_semantic_partitions\', \'reuse_previous_metrics\', \'force\']:\n        if not isinstance(cfg[key], bool):\n            raise ValueError(\'Konfigurasi harus True/False: \' + key)\n    return cfg\n\n\ndef planned_jobs(cfg):\n    per_repeat = len(METHODS)\n    one_off = len(METHODS) - len(STOCHASTIC)\n    counts = {\n        \'LMS_FULL\': one_off + len(STOCHASTIC) * cfg[\'lms_stochastic_runs\'],\n        \'LMS_RESAMPLING\': per_repeat * cfg[\'resample_runs\'],\n        \'LMS_TRAIN\': one_off + len(STOCHASTIC) * cfg[\'holdout_stochastic_runs\'],\n        \'PENDIKOM_TRAIN\': one_off + len(STOCHASTIC) * cfg[\'holdout_stochastic_runs\'],\n        \'BENCHMARK_FULL\': len(cfg[\'benchmark_datasets\']) * (one_off + len(STOCHASTIC) * cfg[\'benchmark_stochastic_runs\']),\n        \'BENCHMARK_RESAMPLING\': len(cfg[\'benchmark_datasets\']) * per_repeat * cfg[\'resample_runs\'],\n    }\n    return {scope: counts[scope] for scope in cfg[\'scopes\']}\n\n\ndef preflight(files, cfg, gd):\n    required = [\'LIB/care_km.py\', \'LIB/experiment_lib.py\', \'LIB/pipeline_data.py\', \'LIB/generalization_data.py\',\n                \'PYTHON/\' + STAGE + \'.ipynb\']\n    if set(cfg[\'scopes\']) & {\'LMS_FULL\', \'LMS_RESAMPLING\'}:\n        required.append(INPUT_LMS)\n    for scope, (dataset, identity, ntr, nte) in TRAIN_INPUTS.items():\n        if scope in cfg[\'scopes\']:\n            required += [f\'OUTPUT/SPLIT/{dataset}/{part}/categorical.csv\' for part in (\'TRAIN\', \'TEST\')]\n    if \'LMS_RESAMPLING\' in cfg[\'scopes\']:\n        required += [f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv\' for r in range(cfg[\'resample_runs\'])]\n    if set(cfg[\'scopes\']) & {\'BENCHMARK_FULL\', \'BENCHMARK_RESAMPLING\'}:\n        for dataset in cfg[\'benchmark_datasets\']:\n            required.append(\'DATA/\' + gd.DATASETS[dataset][\'file\'])\n            if dataset == \'chronic_kidney\':\n                for filename in (\'chronic_kidney_disease_full.arff\', \'chronic_kidney_disease.arff\'):\n                    if (files.root / \'DATA\' / filename).is_file():\n                        required.append(\'DATA/\' + filename)\n            if \'BENCHMARK_RESAMPLING\' in cfg[\'scopes\']:\n                required += [f\'OUTPUT/SPLIT/RESAMPLING/BENCHMARK/19_Generalisasi_Benchmark/{dataset}/rep_{r:02d}.csv\'\n                             for r in range(cfg[\'resample_runs\'])]\n    missing = [p for p in required if not (files.root / p).is_file()]\n    if missing:\n        raise FileNotFoundError(\'Masukan belum lengkap. Lengkapi dari proyek sebelumnya:\\n\' + \'\\n\'.join(missing[:20]))\n    # Colab autosaves rendered cell outputs during long runs. Notebook source\n    # is monitored separately; whole-file output/count changes are not inputs.\n    return {p: files.sha(p) for p in required if not p.endswith(\'.ipynb\')}\n\n\ndef notebook_code(files):\n    notebook = json.loads(files.read(\'PYTHON/\' + STAGE + \'.ipynb\').decode(\'utf-8-sig\'))\n    code = []\n    for cell in notebook[\'cells\']:\n        if cell[\'cell_type\'] != \'code\':\n            continue\n        source = \'\'.join(cell[\'source\'])\n        if \'initializer25_configuration\' in cell.get(\'metadata\', {}).get(\'tags\', []):\n            tree = ast.parse(source)\n            if (len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assign)\n                    or len(tree.body[0].targets) != 1 or not isinstance(tree.body[0].targets[0], ast.Name)\n                    or tree.body[0].targets[0].id != \'KONFIGURASI_25\'\n                    or not isinstance(ast.literal_eval(tree.body[0].value), dict)):\n                raise RuntimeError(\'Sel konfigurasi hanya boleh berisi assignment literal KONFIGURASI_25.\')\n            continue\n        code.append(source)\n    return digest(code)\n\n\ndef same_inputs(files, initial, expected_code=None):\n    for path, expected in initial.items():\n        if files.sha(path) != expected:\n            raise RuntimeError(\'Masukan berubah di tengah eksekusi: \' + path)\n    if expected_code is not None and notebook_code(files) != expected_code:\n        raise RuntimeError(\'Kode notebook berubah selama eksekusi; simpan kode tetap lalu jalankan kembali.\')\n\n\ndef indices(files, relative, n, size):\n    data = pd.read_csv(io.BytesIO(files.read(relative)))\n    idx = data[\'source_row_index\'].to_numpy(dtype=np.int64)\n    if len(idx) != size or len(np.unique(idx)) != size or np.any(idx < 0) or np.any(idx >= n):\n        raise RuntimeError(\'Indeks matched-resampling tidak valid: \' + relative)\n    return idx\n\n\ndef _context(scope, dataset, frame, feature_names, identity, k, sources, rep=None, original_indices=None,\n             test_frame=None, y=None, cfg=None):\n    X = frame[feature_names].astype(str).to_numpy(dtype=object)\n    if len({tuple(row) for row in X}) < k:\n        raise RuntimeError(f\'{scope}/{dataset}: k melebihi jumlah vektor unik.\')\n    ids = frame[identity].astype(str).tolist() if identity in frame else [str(i) for i in range(len(frame))]\n    if len(set(ids)) != len(ids):\n        raise RuntimeError(\'Identitas peserta/baris tidak unik: \' + scope + \'/\' + dataset)\n    idx = np.arange(len(X), dtype=np.int64) if original_indices is None else np.asarray(original_indices, dtype=np.int64)\n    if scope.endswith(\'RESAMPLING\'):\n        runs, base_seed, si_seed = 1, (6161600 if scope == \'LMS_RESAMPLING\' else 130000) + rep, None\n    elif scope == \'BENCHMARK_FULL\':\n        runs, base_seed, si_seed = cfg[\'benchmark_stochastic_runs\'], 120000, 120999\n    elif scope in TRAIN_INPUTS:\n        runs, base_seed, si_seed = cfg[\'holdout_stochastic_runs\'], 4161600, None\n    else:\n        runs, base_seed, si_seed = cfg[\'lms_stochastic_runs\'], 4161600, None\n    Xtest = test_frame[feature_names].astype(str).to_numpy(dtype=object) if test_frame is not None else None\n    test_ids = test_frame[identity].astype(str).tolist() if test_frame is not None else []\n    if set(ids) & set(test_ids):\n        raise RuntimeError(\'Train/test memiliki peserta tumpang tindih.\')\n    result = {\'scope\': scope, \'dataset\': dataset, \'rep\': rep, \'X\': X, \'y\': None if y is None else np.asarray(y, dtype=str),\n              \'features\': feature_names, \'identity\': identity, \'ids\': ids, \'indices\': idx,\n              \'test_X\': Xtest, \'test_ids\': test_ids, \'k\': int(k), \'sources\': sources,\n              \'stochastic_runs\': runs, \'base_seed\': base_seed, \'fixed_si_seed\': si_seed,\n              \'si_sample_size\': None if scope == \'PENDIKOM_TRAIN\' else 2000,\n              \'needs_partition\': scope in TRAIN_INPUTS or (scope.startswith(\'LMS_\') and cfg[\'save_semantic_partitions\'])}\n    result[\'matrix_sha256\'] = digest({\'features\': feature_names, \'rows\': X.tolist(), \'ids\': ids,\n                                      \'indices\': idx.tolist(), \'test_rows\': Xtest.tolist() if Xtest is not None else None,\n                                      \'test_ids\': test_ids, \'classes\': result[\'y\'].tolist() if result[\'y\'] is not None else None})\n    X.flags.writeable = False\n    return result\n\n\ndef contexts(files, cfg, gd, constants, features):\n    scopes = set(cfg[\'scopes\'])\n    if scopes & {\'LMS_FULL\', \'LMS_RESAMPLING\'}:\n        frame = read_frame(files, INPUT_LMS)\n        if len(frame) != 6658 or frame[\'user\'].duplicated().any() or list(frame.columns) != [\'user\'] + features[\'MAIN_FEATURES\']:\n            raise RuntimeError(\'Data LMS full harus 6.658 peserta dengan urutan 13 fitur yang ditetapkan.\')\n        if \'LMS_FULL\' in scopes:\n            yield _context(\'LMS_FULL\', \'LMS\', frame, features[\'MAIN_FEATURES\'], \'user\', 16, [INPUT_LMS], cfg=cfg)\n        if \'LMS_RESAMPLING\' in scopes:\n            for rep in range(cfg[\'resample_runs\']):\n                path = f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv\'\n                idx = indices(files, path, len(frame), int(len(frame) * .8))\n                yield _context(\'LMS_RESAMPLING\', \'LMS\', frame.iloc[idx], features[\'MAIN_FEATURES\'], \'user\', 16,\n                               [INPUT_LMS, path], rep=rep, original_indices=idx, cfg=cfg)\n    for scope, (dataset, identity, ntr, nte) in TRAIN_INPUTS.items():\n        if scope not in scopes:\n            continue\n        train_path, test_path = [f\'OUTPUT/SPLIT/{dataset}/{part}/categorical.csv\' for part in (\'TRAIN\', \'TEST\')]\n        train, test = read_frame(files, train_path), read_frame(files, test_path)\n        if len(train) != ntr or len(test) != nte:\n            raise RuntimeError(\'Jumlah peserta holdout tidak sesuai protokol: \' + dataset)\n        names = features[\'MAIN_FEATURES\'] if dataset == \'LMS\' else features[\'PENDIKOM_USED\']\n        if \'split\' in train and (not train[\'split\'].str.lower().eq(\'train\').all() or not test[\'split\'].str.lower().eq(\'test\').all()):\n            raise RuntimeError(\'Identitas bagian train/test tidak sesuai.\')\n        yield _context(scope, dataset, train, names, identity, 16, [train_path, test_path], test_frame=test, cfg=cfg)\n    if scopes & {\'BENCHMARK_FULL\', \'BENCHMARK_RESAMPLING\'}:\n        for dataset in cfg[\'benchmark_datasets\']:\n            spec = gd.DATASETS[dataset]\n            X, y = gd.load_dataset(files.root / \'DATA\', dataset)\n            if (len(X), X.shape[1], int(pd.Series(y).nunique())) != (spec[\'reference_rows\'], spec[\'reference_features\'], spec[\'reference_k\']):\n                raise RuntimeError(\'Skema benchmark tidak sesuai registry: \' + dataset)\n            frame = X.copy()\n            frame[\'_row_id\'] = np.arange(len(frame)).astype(str)\n            sources = [\'DATA/\' + spec[\'file\']]\n            if dataset == \'chronic_kidney\':\n                for filename in (\'chronic_kidney_disease_full.arff\', \'chronic_kidney_disease.arff\'):\n                    if (files.root / \'DATA\' / filename).is_file():\n                        sources.append(\'DATA/\' + filename)\n            names, k = list(X.columns), int(pd.Series(y).nunique())\n            if \'BENCHMARK_FULL\' in scopes:\n                yield _context(\'BENCHMARK_FULL\', dataset, frame, names, \'_row_id\', k, sources, y=y, cfg=cfg)\n            if \'BENCHMARK_RESAMPLING\' in scopes:\n                for rep in range(cfg[\'resample_runs\']):\n                    path = f\'OUTPUT/SPLIT/RESAMPLING/BENCHMARK/19_Generalisasi_Benchmark/{dataset}/rep_{rep:02d}.csv\'\n                    idx = indices(files, path, len(frame), int(round(len(frame) * .8)))\n                    yield _context(\'BENCHMARK_RESAMPLING\', dataset, frame.iloc[idx], names, \'_row_id\', k, sources + [path],\n                                   rep=rep, original_indices=idx, y=np.asarray(y)[idx], cfg=cfg)\n\n\ndef job_parameters(context, method, run, constants):\n    seed = context[\'base_seed\'] + run\n    return {\'scope\': context[\'scope\'], \'dataset\': context[\'dataset\'], \'rep\': context[\'rep\'], \'run\': run,\n            \'method\': method, \'seed\': seed,\n            \'si_seed\': context[\'fixed_si_seed\'] if context[\'fixed_si_seed\'] is not None else seed,\n            \'si_sample_size\': context[\'si_sample_size\'], \'k\': context[\'k\'],\n            \'max_iter\': constants[\'MAX_ITER\'], \'care_config\': constants[\'FROZEN_CARE_CONFIG\'],\n            \'matrix_sha256\': context[\'matrix_sha256\'], \'needs_partition\': context[\'needs_partition\']}\n\n\ndef _artifact_inventory(root, paths, files):\n    return {str(path.relative_to(root)).replace(\'\\\\\', \'/\'): files.sha(str(path.relative_to(root)).replace(\'\\\\\', \'/\')) for path in paths}\n\n\nclass Legacy:\n    """Reuse only output rows with matching recorded content and relevant protocol."""\n    def __init__(self, files, current_runtime, scientific_hashes):\n        self.files, self.runtime, self.scientific_hashes = files, current_runtime, scientific_hashes\n        path = files.root / \'OUTPUT/KONTROL/status_pipeline.json\'\n        try:\n            records = json.loads(path.read_text(encoding=\'utf-8\'))\n            self.records = {r[\'notebook\']: r for r in records if isinstance(r, dict) and \'notebook\' in r} if isinstance(records, list) else {}\n        except (OSError, UnicodeError, json.JSONDecodeError):\n            self.records = {}\n        self.frames, self.reasons = {}, {}\n\n    def get(self, context, parameters):\n        scope, method = context[\'scope\'], parameters[\'method\']\n        if method == \'BK-Modes\' or context[\'needs_partition\'] or scope not in OLD_SOURCES:\n            return None\n        stem, relative, trial = OLD_SOURCES[scope]\n        record = self.records.get(stem, {})\n        def reject(reason):\n            self.reasons[scope + \'/\' + context[\'dataset\']] = reason\n            return None\n        if record.get(\'status\') != \'selesai\' or not record.get(\'output_manifest\'):\n            return reject(\'catatan isi keluaran sebelumnya belum lengkap\')\n        state = record.get(\'input_manifest\', {})\n        old_runtime = state.get(\'runtime\', {})\n        old_packages = old_runtime.get(\'packages\', {})\n        if any(old_packages.get(k) != v for k, v in self.runtime[\'packages\'].items()) or old_runtime.get(\'python\') != self.runtime[\'python\']:\n            return reject(\'versi Python/paket ilmiah berbeda\')\n        libraries = state.get(\'libraries\', {})\n        for filename, checksum in self.scientific_hashes.items():\n            if libraries.get(filename, {}).get(\'sha256\') != checksum:\n                return reject(\'isi pustaka ilmiah berubah\')\n        recorded_output = record[\'output_manifest\'].get(relative, {})\n        if not (self.files.root / relative).is_file() or self.files.sha(relative) != recorded_output.get(\'sha256\'):\n            return reject(\'isi keluaran lama tidak sesuai catatan\')\n        notebook = \'PYTHON/\' + stem + \'.ipynb\'\n        if not (self.files.root / notebook).is_file():\n            return reject(\'skrip asal tidak tersedia\')\n        old_nb = json.loads(self.files.read(notebook).decode(\'utf-8-sig\'))\n        old_code = [\'\'.join(c[\'source\']) for c in old_nb[\'cells\'] if c[\'cell_type\'] == \'code\']\n        if digest(old_code) != state.get(\'notebooks\', {}).get(notebook):\n            return reject(\'skrip asal berubah\')\n        known = {}\n        known.update(state.get(\'data\', {})); known.update(state.get(\'preprocessed\', {}))\n        known.update(record.get(\'output_manifest\', {}))\n        for dependency in state.get(\'dependencies\', {}).values():\n            known.update(dependency)\n        for source in context[\'sources\']:\n            if known.get(source, {}).get(\'sha256\') != self.files.sha(source):\n                return reject(\'isi data atau indeks sumber berubah/tidak tercatat\')\n        if relative not in self.frames:\n            self.frames[relative] = pd.read_csv(io.BytesIO(self.files.read(relative)), float_precision=\'round_trip\')\n        frame = self.frames[relative]\n        mask = frame[\'method\'].eq(method) & pd.to_numeric(frame[trial], errors=\'coerce\').eq(context[\'rep\'] if context[\'rep\'] is not None else parameters[\'run\'])\n        if scope.startswith(\'BENCHMARK\'):\n            mask &= frame[\'dataset\'].eq(context[\'dataset\'])\n        matched = frame.loc[mask]\n        if len(matched) != 1:\n            return reject(\'baris hasil tidak unik atau belum tersedia\')\n        row = matched.iloc[0].to_dict()\n        if int(row.get(\'k\', -1)) != parameters[\'k\'] or int(row.get(\'seed\', -1)) != parameters[\'seed\']:\n            return reject(\'k/seed tidak sesuai protokol\')\n        if context[\'rep\'] is not None:\n            sample_hash = binary_hash(np.asarray(context[\'indices\'], dtype=np.int64).tobytes())[:16]\n            if row.get(\'sample_hash\') != sample_hash:\n                return reject(\'anggota matched-resample berbeda\')\n        if any(not np.isfinite(float(row.get(field, np.nan))) for field in METRICS):\n            return reject(\'metrik lama tidak valid\')\n        evidence = {\'file\': relative, \'sha256\': self.files.sha(relative), \'stage\': stem,\n                    \'source_notebook\': notebook, \'source_code_sha256\': digest(old_code),\n                    \'source_runtime\': old_runtime}\n        return clean(row), evidence\n\n\ndef fit_job(core, bk, context, parameters):\n    X, k = context[\'X\'], parameters[\'k\']\n    t0 = time.perf_counter()\n    if parameters[\'method\'] == \'BK-Modes\':\n        detail = bk.bk_modes_init(X, k, max_iter=parameters[\'max_iter\'], return_details=True)\n        init = detail[\'modes\']\n    else:\n        detail = None\n        init = core.init_by_name(parameters[\'method\'], X, k, seed=parameters[\'seed\'], care_config=parameters[\'care_config\'])\n    t1 = time.perf_counter()\n    fit = core.kmodes_fit(X, init, max_iter=parameters[\'max_iter\'])\n    t2 = time.perf_counter()\n    flags = []\n    counts = np.bincount(fit[\'labels\'], minlength=k)\n    if (counts == 0).any(): flags.append(\'empty_final_cluster\')\n    initial_cost = float(core.initial_cost(X, init))\n    if fit[\'cost\'] > initial_cost + 1e-12: flags.append(\'objective_worsened\')\n    if int(fit[\'iterations\']) < 1: flags.append(\'nonpositive_iterations\')\n    try:\n        metrics = core.evaluate_clustering(X, context[\'y\'], fit, init,\n                                           si_sample_size=parameters[\'si_sample_size\'], si_seed=parameters[\'si_seed\'])\n    except ValueError as error:\n        metrics = {\'initial_cost\': initial_cost, \'final_cost\': float(fit[\'cost\']),\n                   \'iterations\': int(fit[\'iterations\']), \'dbi\': None, \'si\': None}\n        flags.append(\'metric_error: \' + str(error))\n    for field in METRICS:\n        if metrics.get(field) is None or not np.isfinite(float(metrics[field])):\n            flags.append(\'nonfinite_\' + field)\n    row = {**metrics, \'init_sec\': t1-t0, \'cluster_sec\': t2-t1, \'total_sec\': t2-t0,\n           \'valid\': not flags, \'failure_flags\': \'; \'.join(flags), \'origin\': \'COMPUTED\',\n           \'runtime_comparable\': False}\n    return clean(row), init, fit, detail\n\n\ndef store_partition(folder, context, parameters, init, fit, detail, core):\n    paths = []\n    for filename, array in [(\'initial_modes.csv\', init), (\'final_modes.csv\', fit[\'modes\'])]:\n        frame = pd.DataFrame(array, columns=context[\'features\'])\n        frame.insert(0, \'cluster_id\', np.arange(parameters[\'k\']))\n        path = folder / filename; save_csv(path, frame); paths.append(path)\n    assignments = pd.DataFrame({\'source_row_index\': context[\'indices\'], \'identity\': context[\'ids\'], \'cluster_id\': fit[\'labels\']})\n    path = folder / \'train_assignments.csv\'; save_csv(path, assignments); paths.append(path)\n    if context[\'test_X\'] is not None:\n        labels, distance = core.assign_nearest(context[\'test_X\'], fit[\'modes\'])\n        path = folder / \'test_assignments.csv\'\n        save_csv(path, pd.DataFrame({\'identity\': context[\'test_ids\'], \'cluster_id\': labels, \'distance\': distance}))\n        paths.append(path)\n    if detail is not None:\n        path = folder / \'bisection_trace.json\'; save_json(path, detail[\'split_trace\']); paths.append(path)\n    return paths\n\n\ndef execute_job(root, output, control, context, method, run, constants, core, bk,\n                code_state, current_runtime, files, legacy, cfg):\n    parameters = job_parameters(context, method, run, constants)\n    signature = digest({\'parameters\': parameters, \'code\': code_state, \'runtime\': current_runtime})\n    identity = {key: parameters[key] for key in (\'scope\', \'dataset\', \'rep\', \'run\', \'method\')}\n    key = digest(identity)[:24]\n    status_file = control / \'JOBS\' / (key + \'.json\')\n    if status_file.is_file() and not cfg[\'force\']:\n        previous = read_checkpoint(status_file)\n        checksum = previous.pop(\'record_sha256\', \'\')\n        try:\n            valid_record = checksum == digest(previous)\n        except (TypeError, ValueError):\n            valid_record = False\n        good_outputs = valid_record and all((root / path).is_file() and files.sha(path) == checksum for path, checksum in previous.get(\'outputs\', {}).items())\n        evidence = previous.get(\'legacy_evidence\')\n        good_evidence = not evidence or ((root / evidence[\'file\']).is_file() and files.sha(evidence[\'file\']) == evidence[\'sha256\'])\n        if valid_record and previous.get(\'status\') == \'selesai\' and previous.get(\'signature\') == signature and good_outputs and good_evidence:\n            return previous[\'row\'], \'CACHE\', previous\n    reused = legacy.get(context, parameters) if cfg[\'reuse_previous_metrics\'] and not cfg[\'force\'] else None\n    folder = output / \'BANK\' / context[\'scope\'] / context[\'dataset\'] / (\'r\' + str(context[\'rep\'] if context[\'rep\'] is not None else run).zfill(2)) / re.sub(r\'[^A-Za-z0-9]+\', \'_\', method.replace(\'++\', \'_PP\')).strip(\'_\')\n    paths, evidence = [], None\n    if reused is not None:\n        row, evidence = reused\n        row.update(valid=True, failure_flags=\'\', origin=\'REUSED_VERIFIED\', runtime_comparable=False)\n        disposition = \'REUSE\'\n    else:\n        row, init, fit, detail = fit_job(core, bk, context, parameters)\n        # Retain all newly computed partitions, including benchmark runs.\n        paths = store_partition(folder, context, parameters, init, fit, detail, core)\n        disposition = \'FIT\'\n    row.update(scope=context[\'scope\'], dataset=context[\'dataset\'], method=method, run=run,\n               rep=context[\'rep\'], seed=parameters[\'seed\'], si_seed=parameters[\'si_seed\'],\n               k=parameters[\'k\'], sample_n=len(context[\'X\']), matrix_sha256=context[\'matrix_sha256\'],\n               sample_hash=binary_hash(np.asarray(context[\'indices\'], dtype=np.int64).tobytes())[:16],\n               partition_available=bool(paths), features=len(context[\'features\']))\n    record = {\'schema\': 1, \'status\': \'selesai\', \'signature\': signature, \'identity\': identity,\n              \'parameters\': parameters, \'row\': clean(row), \'outputs\': _artifact_inventory(root, paths, files),\n              \'legacy_evidence\': evidence, \'finished_utc\': datetime.now(timezone.utc).isoformat()}\n    record[\'record_sha256\'] = digest(record)\n    save_json(status_file, record)\n    return record[\'row\'], disposition, record\n\n\ndef summarize(raw):\n    rows = []\n    for (scope, dataset, method), group in raw.groupby([\'scope\', \'dataset\', \'method\'], sort=False):\n        valid = group.loc[group[\'valid\'].eq(True)]\n        row = {\'scope\': scope, \'dataset\': dataset, \'method\': method, \'n_runs\': len(group),\n               \'n_valid\': len(valid), \'n_failures\': len(group)-len(valid),\n               \'partition_runs\': int(group[\'partition_available\'].eq(True).sum())}\n        for metric in METRICS:\n            values = pd.to_numeric(valid[metric], errors=\'coerce\')\n            row[metric + \'_mean\'], row[metric + \'_sd\'] = values.mean(), values.std(ddof=1)\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef exact_signed_rank(differences, zero_tolerance=1e-12):\n    differences = np.asarray(differences, dtype=float)\n    if not np.isfinite(differences).all():\n        raise ValueError(\'Paired differences mengandung nilai tidak valid.\')\n    nonzero = np.abs(differences) > zero_tolerance\n    values = differences[nonzero]\n    if not len(values):\n        return {\'statistic\': 0.0, \'p_value\': 1.0, \'rank_biserial\': 0.0, \'n_nonzero\': 0,\n                \'n_zero\': len(differences), \'absolute_ranks\': [], \'positive_rank_sum\': 0.0, \'negative_rank_sum\': 0.0}\n    ranks = pd.Series(np.abs(values)).rank(method=\'average\').to_numpy(float)\n    positive, negative = float(ranks[values > 0].sum()), float(ranks[values < 0].sum())\n    observed = min(positive, negative)\n    count = 0\n    for signs in itertools.product((0, 1), repeat=len(values)):\n        pos = float(np.dot(signs, ranks))\n        count += min(pos, ranks.sum()-pos) <= observed + 1e-12\n    return {\'statistic\': observed, \'p_value\': count / (2 ** len(values)),\n            \'rank_biserial\': (positive-negative)/(positive+negative), \'n_nonzero\': len(values),\n            \'n_zero\': int((~nonzero).sum()), \'absolute_ranks\': ranks.tolist(),\n            \'positive_rank_sum\': positive, \'negative_rank_sum\': negative}\n\n\ndef holm(values):\n    order = np.argsort(values, kind=\'stable\')\n    answer, running = np.zeros(len(values)), 0.0\n    for rank, index in enumerate(order):\n        running = max(running, (len(values)-rank)*values[index])\n        answer[index] = min(1.0, running)\n    return answer\n\n\ndef benchmark_statistics(summary, cfg, gd, output):\n    from scipy.stats import friedmanchisquare\n    ranks, friedman, pairwise, audit = [], [], [], []\n    notes = []\n    for scope in [\'BENCHMARK_FULL\', \'BENCHMARK_RESAMPLING\']:\n        frame = summary.loc[summary[\'scope\'].eq(scope)]\n        if frame.empty:\n            notes.append(scope + \': belum dijalankan\')\n            continue\n        if scope == \'BENCHMARK_RESAMPLING\' and cfg[\'resample_runs\'] != 30:\n            notes.append(scope + \': pengaturan diagnostik kurang dari 30 resample; statistik utama tidak dihitung\')\n            continue\n        if scope == \'BENCHMARK_FULL\' and cfg[\'benchmark_stochastic_runs\'] != 10:\n            notes.append(scope + \': pengaturan diagnostik bukan 10 run stokastik; statistik utama tidak dihitung\')\n            continue\n        complete = set(frame.dataset) == set(gd.DATASET_ORDER) and len(frame) == len(gd.DATASET_ORDER)*len(METHODS)\n        expected_runs = frame.method.map(lambda method: 30 if scope == \'BENCHMARK_RESAMPLING\'\n                                         else (10 if method in STOCHASTIC else 1))\n        if not complete or (frame.n_failures > 0).any() or not frame.n_runs.eq(expected_runs).all():\n            notes.append(scope + \': statistik sepuluh dataset belum lengkap atau ada kegagalan; tidak dihitung paksa\')\n            continue\n        for metric in METRICS:\n            matrix = frame.pivot(index=\'dataset\', columns=\'method\', values=metric+\'_mean\').reindex(index=gd.DATASET_ORDER, columns=METHODS)\n            if not np.isfinite(matrix.to_numpy(float)).all():\n                notes.append(scope + \'/\' + metric + \': metrik tidak lengkap\')\n                continue\n            ranking = matrix.rank(axis=1, ascending=metric != \'si\', method=\'average\')\n            for dataset in matrix.index:\n                for method in METHODS:\n                    ranks.append({\'source\': scope, \'metric\': metric, \'dataset\': dataset, \'method\': method,\n                                  \'value\': matrix.loc[dataset, method], \'rank\': ranking.loc[dataset, method]})\n            if (matrix.nunique(axis=1) == 1).all():\n                stat, p = np.nan, np.nan\n                notes.append(scope + \'/\' + metric + \': semua metode seri; Friedman tidak terdefinisi\')\n            else:\n                stat, p = friedmanchisquare(*(matrix[method].to_numpy(float) for method in METHODS))\n            friedman.append({\'source\': scope, \'metric\': metric, \'friedman_chi2\': stat, \'p_value\': p,\n                             \'kendall_w\': stat/(len(matrix)*(len(METHODS)-1)), \'n_datasets\': len(matrix), \'n_methods\': len(METHODS)})\n            if scope != \'BENCHMARK_RESAMPLING\' or metric not in (\'dbi\', \'si\'):\n                continue\n            family = []\n            for method in [m for m in METHODS if m != \'CARE-KM\']:\n                difference = matrix[\'CARE-KM\'] - matrix[method] if metric == \'si\' else matrix[method] - matrix[\'CARE-KM\']\n                result = exact_signed_rank(difference.to_numpy(float))\n                for dataset, value in difference.items():\n                    audit.append({\'metric\': metric, \'baseline\': method, \'dataset\': dataset,\n                                  \'care_value\': matrix.loc[dataset, \'CARE-KM\'], \'baseline_value\': matrix.loc[dataset, method],\n                                  \'improvement_favoring_care\': value, \'zero\': abs(value) <= 1e-12})\n                family.append({\'metric\': metric, \'baseline\': method, \'wins\': int((difference > 1e-12).sum()),\n                               \'ties\': int((difference.abs() <= 1e-12).sum()), \'losses\': int((difference < -1e-12).sum()),\n                               **{k: v for k, v in result.items() if k != \'absolute_ranks\'},\n                               \'p_method\': \'exact enumeration of nonzero signed-rank signs\', \'zero_method\': \'discard abs(diff)<=1e-12\',\n                               \'absolute_rank_ties\': \'average\', \'alternative\': \'two-sided\'})\n            adjusted = holm([row[\'p_value\'] for row in family])\n            for row, value in zip(family, adjusted):\n                row[\'holm_p\'] = value; pairwise.append(row)\n    folder = output / \'STATISTIK_BENCHMARK\'\n    save_csv(folder / \'01_dataset_values_and_ranks.csv\', pd.DataFrame(ranks, columns=[\'source\',\'metric\',\'dataset\',\'method\',\'value\',\'rank\']))\n    rank_frame = pd.DataFrame(ranks)\n    average = (rank_frame.groupby([\'source\', \'metric\', \'method\'], as_index=False)[\'rank\'].mean().rename(columns={\'rank\': \'average_rank\'})\n               if len(rank_frame) else pd.DataFrame(columns=[\'source\', \'metric\', \'method\', \'average_rank\']))\n    save_csv(folder / \'02_average_ranks.csv\', average)\n    save_csv(folder / \'03_friedman_kendall.csv\', pd.DataFrame(friedman, columns=[\'source\',\'metric\',\'friedman_chi2\',\'p_value\',\'kendall_w\',\'n_datasets\',\'n_methods\']))\n    save_csv(folder / \'04_paired_wilcoxon_holm.csv\', pd.DataFrame(pairwise) if pairwise else pd.DataFrame(columns=[\'metric\',\'baseline\',\'wins\',\'ties\',\'losses\',\'statistic\',\'p_value\',\'rank_biserial\',\'holm_p\']))\n    audit_frame = pd.DataFrame(audit)\n    if len(audit_frame):\n        audit_frame[\'absolute_rank\'] = audit_frame.groupby([\'metric\', \'baseline\'])[\'improvement_favoring_care\'].transform(\n            lambda x: x.abs().where(x.abs() > 1e-12).rank(method=\'average\'))\n        audit_frame[\'signed_rank\'] = np.sign(audit_frame.improvement_favoring_care) * audit_frame.absolute_rank\n    save_csv(folder / \'05_paired_differences_and_ranks.csv\', audit_frame if len(audit_frame) else pd.DataFrame(columns=[\'metric\',\'baseline\',\'dataset\',\'care_value\',\'baseline_value\',\'improvement_favoring_care\',\'zero\',\'absolute_rank\',\'signed_rank\']))\n    save_json(folder / \'status_statistik.json\', {\'notes\': notes, \'independent_units\': \'ten datasets\',\n              \'primary_source\': \'means of thirty matched resamples within each dataset\',\n              \'multiple_testing\': \'Holm, seven CARE-KM contrasts separately for DBI and SI\',\n              \'runtime_ranks\': \'not calculated from mixed cached/new timing measurements\'})\n    return notes\n\n\ndef run_comparison(root, bk, verify_cases, configuration, code_sources):\n    root = Path(root).expanduser().resolve()\n    files = Files(root)\n    core, gd, constants, features = load_project(files)\n    cfg = validate_config(configuration, gd)\n    watched = preflight(files, cfg, gd)\n    output = root / \'OUTPUT/TAMBAHAN\' / STAGE\n    control = root / \'OUTPUT/TAMBAHAN/KONTROL\' / STAGE\n    output.mkdir(parents=True, exist_ok=True); control.mkdir(parents=True, exist_ok=True)\n    # Retire the previous pilot-only notebook safely, keeping its contents.\n    old_notebook = root / \'PYTHON\' / (OLD_STAGE + \'.ipynb\')\n    if old_notebook.is_file():\n        archive = control / \'ARSIP_NOTEBOOK\' / (OLD_STAGE + \'_\' + uuid.uuid4().hex[:8] + \'.ipynb\')\n        archive.parent.mkdir(parents=True, exist_ok=True)\n        old_notebook.rename(archive)\n        print(\'Notebook pilot lama dipindahkan ke arsip tahap tambahan.\')\n    current_runtime = runtime()\n    hashes = {name: files.sha(name) for name in [\'LIB/care_km.py\', \'LIB/experiment_lib.py\', \'LIB/generalization_data.py\', \'LIB/pipeline_data.py\']}\n    code_state = {\'project_libraries\': hashes, \'embedded_sources\': {k: binary_hash(v.encode(\'utf-8\')) for k, v in code_sources.items()},\n                  \'notebook_code\': notebook_code(files)}\n    verify_path = control / \'verifikasi_kecil.json\'\n    verify_key = digest({\'code\': code_state, \'runtime\': current_runtime})\n    check_file = output / \'VERIFIKASI/01_kasus_kecil.csv\'\n    saved_verification = read_checkpoint(verify_path)\n    if not cfg[\'force\'] and saved_verification.get(\'key\') == verify_key and check_file.is_file() and files.sha(str(check_file.relative_to(root)).replace(\'\\\\\',\'/\')) == saved_verification.get(\'output_sha256\'):\n        checks = pd.read_csv(check_file)\n        print(\'Verifikasi kecil: gunakan hasil yang isinya terverifikasi sama.\')\n    else:\n        checks = verify_cases(bk)\n        save_csv(check_file, checks)\n        save_json(verify_path, {\'key\': verify_key, \'output_sha256\': binary_hash(check_file.read_bytes())})\n    if not checks[\'status\'].eq(\'LULUS\').all():\n        raise RuntimeError(\'Verifikasi kecil belum lulus.\')\n    print(\'Verifikasi kecil:\', len(checks), \'pemeriksaan LULUS.\')\n    save_csv(output / \'PROTOKOL/01_daftar_initializer.csv\', pd.DataFrame([{\'method\': m, \'stochastic\': m in STOCHASTIC} for m in METHODS]))\n    save_json(output / \'PROTOKOL/02_konfigurasi.json\', {\'config\': cfg, \'constants\': constants,\n              \'runtime\': current_runtime, \'code\': code_state, \'input_hashes\': watched,\n              \'bk_policy\': bk.IMPLEMENTATION_POLICY,\n              \'semantic_scope\': \'saved partitions/assignments; semantic rules and ILS correspondence are evaluated in the next experiment\',\n              \'runtime_scope\': \'stored timings have origin tags; no comparative-runtime claim from mixed sessions\',\n              \'stochastic_references\': \'run 0 is the declared reference; no best-run selection\'})\n    legacy = Legacy(files, current_runtime, hashes)\n    rows, artifacts, counts = [], [], {\'CACHE\': 0, \'REUSE\': 0, \'FIT\': 0}\n    expected_jobs = planned_jobs(cfg)\n    print(\'Jumlah run terjadwal:\', sum(expected_jobs.values()), \'|\', expected_jobs, flush=True)\n    context_number = 0\n    started = time.time()\n    try:\n        for context in contexts(files, cfg, gd, constants, features):\n            same_inputs(files, watched, code_state[\'notebook_code\'])\n            context_number += 1\n            print(f"[{context_number}] {context[\'scope\']} / {context[\'dataset\']} / rep={context[\'rep\']} | n={len(context[\'X\'])}, k={context[\'k\']}", flush=True)\n            for method in METHODS:\n                runs = context[\'stochastic_runs\'] if method in STOCHASTIC else 1\n                for run in range(runs):\n                    row, disposition, record = execute_job(root, output, control, context, method, run,\n                                                          constants, core, bk, code_state, current_runtime, files, legacy, cfg)\n                    rows.append(row); counts[disposition] += 1\n                    artifacts.append({\'scope\': context[\'scope\'], \'dataset\': context[\'dataset\'], \'rep\': context[\'rep\'],\n                                      \'run\': run, \'method\': method, \'partition_available\': row[\'partition_available\'],\n                                      \'files\': json.dumps(list(record.get(\'outputs\', {})), ensure_ascii=False)})\n                    if disposition == \'FIT\':\n                        print(\'  \', method, \'run\', run, \': dihitung dan disimpan\', flush=True)\n            # Export partial tables after every complete context; JOBS remain\n            # the authoritative restart point even if the session is interrupted.\n            save_csv(output / \'01_hasil_per_run_8_metode.csv\', pd.DataFrame(rows))\n            save_json(control / \'status_eksekusi.json\', {\'status\': \'berjalan\', \'completed_jobs\': len(rows),\n                      \'counts\': counts, \'last_scope\': context[\'scope\'], \'last_dataset\': context[\'dataset\']})\n        same_inputs(files, watched, code_state[\'notebook_code\'])\n        raw = pd.DataFrame(rows)\n        if len(raw) != sum(expected_jobs.values()):\n            raise RuntimeError(\'Jumlah run selesai tidak sesuai konfigurasi.\')\n        summary = summarize(raw)\n        save_csv(output / \'02_ringkasan_8_metode.csv\', summary)\n        save_csv(output / \'03_peta_mode_dan_assignment.csv\', pd.DataFrame(artifacts))\n        save_csv(output / \'04_failure_audit.csv\', raw[[\'scope\',\'dataset\',\'method\',\'run\',\'rep\',\'valid\',\'failure_flags\',\'origin\']])\n        for scope in cfg[\'scopes\']:\n            save_csv(output / scope / \'hasil_per_run.csv\', raw.loc[raw.scope.eq(scope)])\n            save_csv(output / scope / \'ringkasan.csv\', summary.loc[summary.scope.eq(scope)])\n        notes = benchmark_statistics(summary, cfg, gd, output)\n        full_protocol = (set(cfg[\'scopes\']) == set(SCOPES) and cfg[\'benchmark_datasets\'] == gd.DATASET_ORDER\n                         and cfg[\'resample_runs\'] == 30 and cfg[\'lms_stochastic_runs\'] == 15\n                         and cfg[\'holdout_stochastic_runs\'] == 15 and cfg[\'benchmark_stochastic_runs\'] == 10\n                         and cfg[\'save_semantic_partitions\'])\n        status = {\'status\': \'selesai\' if raw[\'valid\'].eq(True).all() else \'selesai_dengan_kegagalan_algoritmik\',\n                  \'completed_jobs\': len(raw), \'n_valid\': int(raw[\'valid\'].eq(True).sum()),\n                  \'counts\': counts, \'method_count\': len(METHODS), \'full_protocol\': full_protocol,\n                  \'expected_jobs_by_scope\': expected_jobs,\n                  \'elapsed_sec\': time.time()-started, \'statistics_notes\': notes,\n                  \'reuse_reasons\': legacy.reasons, \'output\': str(output),\n                  \'semantic_ILS_evaluation\': \'not part of this initializer/partition-bank experiment\'}\n        save_json(control / \'status_eksekusi.json\', status)\n        print(\'Tahap 25 selesai:\', len(raw), \'run,\', len(METHODS), \'metode. Cache/reuse/fit:\', counts)\n        print(\'Hasil:\', output)\n        return {\'summary\': summary, \'status\': status, \'output\': str(output)}\n    except BaseException as error:\n        save_json(control / \'status_eksekusi.json\', {\'status\': \'terhenti\', \'completed_jobs\': len(rows), \'counts\': counts,\n                  \'error\': f\'{type(error).__name__}: {error}\', \'resume\': \'rerun the same notebook/configuration; per-job contents are checked\'})\n        raise\n'

_h25 = types.ModuleType('_frozen_helpers25')
exec(compile(HELPER25_SOURCE, '_frozen_helpers25', 'exec'), _h25.__dict__)


In [9]:
BK25_SOURCE = '"""Python reimplementation of Algorithms 8–9, arXiv:2502.07081v2.\n\nThis module returns initial modes only. The final K-Modes optimization is\ndelegated to the unchanged optimizer of the host experiment pipeline.\nPaper-unspecified tie/empty/cap policies are explicit in IMPLEMENTATION_POLICY.\n"""\nimport numpy as np\n\nBK_IMPLEMENTATION_VERSION = \'bk-modes-2025-python-1.0.0\'\nPAPER_URL = \'https://arxiv.org/abs/2502.07081v2\'\nIMPLEMENTATION_POLICY = {\n    \'source\': PAPER_URL,\n    \'publication_status\': \'arXiv preprint, v2, 15 February 2025\',\n    \'implementation\': \'Python reimplementation; not author-supplied C code\',\n    \'cluster_selection\': \'largest within-leaf sum of SMD; first-created leaf on a tie\',\n    \'two_modes_initial_centers\': \'mode of selected leaf and first farthest source row\',\n    \'two_modes_assignment_tie\': \'second child, as in Algorithm 9\',\n    \'initial_categorical_mode_tie\': \'lexicographically smallest string\',\n    \'updated_categorical_mode_tie\': \'retain previous category if modal; otherwise lexicographic\',\n    \'farthest_row_tie\': \'first in the preserved source-row order\',\n    \'duplicate_rows\': \'retained with their original frequencies\',\n    \'identical_leaf\': \'not bisected; select another splittable leaf\',\n    \'empty_child\': \'move first farthest row from a non-singleton donor, then update modes; logged\',\n    \'inner_stopping\': \'child memberships unchanged; explicit iteration cap raises an error\',\n    \'final_kmodes\': \'host care_km.kmodes_fit, unchanged; host tie and empty-cluster rules\',\n    \'timing_scope\': \'all leaf selection, bisection, and mode construction belong to initialization\',\n    \'number_of_centers\': \'k leaf modes; coincident leaf modes are reported, not silently deduplicated\',\n    \'determinism\': \'fixed implementation, string categories, input order, and parameters\',\n}\n\n\ndef _validate_X(X):\n    X = np.asarray(X, dtype=object)\n    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:\n        raise ValueError(\'X harus berupa matriks kategorikal dengan baris dan fitur yang tidak kosong.\')\n    if any(not isinstance(value, (str, np.str_)) for value in X.flat):\n        raise TypeError(\'Kategori X harus berupa string yang sudah disiapkan secara konsisten.\')\n    if any(value == \'\' for value in X.flat):\n        raise ValueError(\'Kategori kosong harus ditangani sebelum inisialisasi.\')\n    return X\n\n\ndef _smd(X, centers):\n    """An n-by-k SMD matrix; avoid an n-by-k-by-d temporary."""\n    result = np.zeros((len(X), len(centers)), dtype=np.int64)\n    for feature in range(X.shape[1]):\n        result += X[:, feature, None] != centers[None, :, feature]\n    return result\n\n\ndef _mode(X, previous=None):\n    center = []\n    for feature in range(X.shape[1]):\n        values, counts = np.unique(X[:, feature], return_counts=True)\n        winners = values[counts == counts.max()]\n        old = previous[feature] if previous is not None else None\n        center.append(old if old in winners else winners[0])\n    return np.asarray(center, dtype=object)\n\n\ndef _assign_two(X, centers):\n    distances = _smd(X, centers)\n    # The strict comparison and second-child tie implement Algorithm 9.\n    labels = np.where(distances[:, 0] < distances[:, 1], 0, 1)\n    return labels.astype(np.int64), distances\n\n\ndef _two_modes(X, max_iter):\n    first = _mode(X)\n    distances = _smd(X, first[None, :])[:, 0]\n    farthest = int(np.argmax(distances))\n    if int(distances[farthest]) == 0:\n        raise ValueError(\'Leaf dengan satu vektor unik tidak dapat dibelah.\')\n    centers = np.asarray([first, X[farthest].copy()], dtype=object)\n    previous_labels = None\n    repairs = 0\n    history = []\n    for iteration in range(1, max_iter + 1):\n        labels, matrix = _assign_two(X, centers)\n        counts = np.bincount(labels, minlength=2)\n        for empty in np.flatnonzero(counts == 0):\n            assigned_distance = matrix[np.arange(len(X)), labels]\n            candidates = [int(i) for i in np.argsort(-assigned_distance, kind=\'stable\')\n                          if counts[labels[i]] > 1]\n            if not candidates:\n                raise RuntimeError(\'Tidak ada donor untuk memperbaiki child kosong.\')\n            selected = candidates[0]\n            counts[labels[selected]] -= 1\n            labels[selected] = int(empty)\n            counts[empty] += 1\n            repairs += 1\n        centers = np.asarray([_mode(X[labels == child], centers[child]) for child in range(2)], dtype=object)\n        objective = int(_smd(X, centers)[np.arange(len(X)), labels].sum())\n        history.append(objective)\n        if len(history) > 1 and history[-1] > history[-2]:\n            raise RuntimeError(\'Objective Two-Modes meningkat pada pembaruan.\')\n        if previous_labels is not None and np.array_equal(labels, previous_labels):\n            return {\'labels\': labels, \'centers\': centers, \'iterations\': iteration,\n                    \'empty_repairs\': repairs, \'objective_history\': history,\n                    \'initial_farthest_local_row\': farthest}\n        previous_labels = labels.copy()\n    raise RuntimeError(f\'Two-Modes belum konvergen setelah {max_iter} iterasi; hasil parsial tidak dipakai.\')\n\n\ndef bk_modes_init(X, k, max_iter=100, return_details=False):\n    """Return k initial leaf modes without running final global K-Modes.\n\n    ``return_details=True`` also returns the leaf memberships and split trace.\n    Categories must be explicit strings, as in the host pipeline datasets.\n    The source paper leaves several implementation choices unspecified;\n    those choices are recorded in IMPLEMENTATION_POLICY rather than hidden.\n    """\n    X = _validate_X(X)\n    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1:\n        raise ValueError(\'k harus berupa bilangan bulat positif.\')\n    if isinstance(max_iter, bool) or not isinstance(max_iter, (int, np.integer)) or max_iter < 1:\n        raise ValueError(\'max_iter harus berupa bilangan bulat positif.\')\n    k, max_iter = int(k), int(max_iter)\n    unique_rows = len({tuple(row) for row in X})\n    if k > unique_rows:\n        raise ValueError(f\'k={k} melebihi jumlah vektor kategorikal unik ({unique_rows}).\')\n\n    def node(identifier, indices, center=None):\n        center = _mode(X[indices]) if center is None else np.asarray(center, dtype=object).copy()\n        cost = int(_smd(X[indices], center[None, :]).sum())\n        return {\'node_id\': int(identifier), \'indices\': indices, \'center\': center, \'cost\': cost}\n\n    root = node(0, np.arange(len(X), dtype=np.int64))\n    leaves = {0: root}\n    next_id = 1\n    trace = []\n    for step in range(1, k):\n        splittable = [leaf for leaf in leaves.values() if leaf[\'cost\'] > 0]\n        if not splittable:\n            raise RuntimeError(\'Jumlah leaf yang diminta tidak tercapai.\')\n        parent = min(splittable, key=lambda leaf: (-leaf[\'cost\'], leaf[\'node_id\']))\n        local = _two_modes(X[parent[\'indices\']], max_iter)\n        children = []\n        for child in range(2):\n            indices = parent[\'indices\'][local[\'labels\'] == child]\n            if not len(indices):\n                raise RuntimeError(\'Two-Modes menghasilkan child kosong.\')\n            children.append(node(next_id + child, indices, local[\'centers\'][child]))\n        after_cost = sum(child[\'cost\'] for child in children)\n        if after_cost > parent[\'cost\']:\n            raise RuntimeError(\'Biseksi meningkatkan jumlah SMD dalam leaf.\')\n        trace.append({\'step\': step, \'parent_node\': parent[\'node_id\'],\n                      \'parent_rows\': int(len(parent[\'indices\'])), \'parent_cost\': parent[\'cost\'],\n                      \'child_nodes\': [child[\'node_id\'] for child in children],\n                      \'child_rows\': [int(len(child[\'indices\'])) for child in children],\n                      \'child_costs\': [child[\'cost\'] for child in children],\n                      \'parent_mode\': parent[\'center\'].tolist(),\n                      \'farthest_source_row\': int(parent[\'indices\'][local[\'initial_farthest_local_row\']]),\n                      \'two_modes_iterations\': local[\'iterations\'],\n                      \'empty_repairs\': local[\'empty_repairs\'],\n                      \'objective_history\': local[\'objective_history\']})\n        del leaves[parent[\'node_id\']]\n        for child in children:\n            leaves[child[\'node_id\']] = child\n        next_id += 2\n    ordered = [leaves[key] for key in sorted(leaves)]\n    modes = np.asarray([leaf[\'center\'] for leaf in ordered], dtype=object)\n    labels = np.full(len(X), -1, dtype=np.int64)\n    for label, leaf in enumerate(ordered):\n        labels[leaf[\'indices\']] = label\n    if np.any(labels < 0) or len(ordered) != k:\n        raise RuntimeError(\'Keanggotaan leaf belum lengkap.\')\n    details = {\'modes\': modes, \'leaf_labels\': labels, \'split_trace\': trace,\n               \'initial_global_mode_cost\': root[\'cost\'],\n               \'leaf_partition_cost\': sum(leaf[\'cost\'] for leaf in ordered),\n               \'initial_assignment_cost\': int(_smd(X, modes).min(axis=1).sum()),\n               \'unique_initial_modes\': len({tuple(row) for row in modes}),\n               \'total_two_modes_iterations\': sum(row[\'two_modes_iterations\'] for row in trace),\n               \'empty_repairs\': sum(row[\'empty_repairs\'] for row in trace),\n               \'version\': BK_IMPLEMENTATION_VERSION}\n    return details if return_details else modes\n'

_bk25 = types.ModuleType('_frozen_bk25')
exec(compile(BK25_SOURCE, '_frozen_bk25', 'exec'), _bk25.__dict__)

RUNTIME31_SOURCE = '"""Same-matrix eight-method runtime with matched source seeds and failure audit."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport ast,io,json,math,os,platform,time,types,uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'31_Benchmark_Runtime_dan_Failure_Audit\'\nU25=\'OUTPUT/TAMBAHAN/25_Perbandingan_Initializer_Terintegrasi\'\nC25=\'OUTPUT/TAMBAHAN/KONTROL/25_Perbandingan_Initializer_Terintegrasi\'\nDETERMINISTIC=[\'Cao\',\'Bai\',\'CARE-KM\',\'BK-Modes\']\nSTOCHASTIC={\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\nTIMING_METHODS=DETERMINISTIC+[\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\']\nSOURCE_SEEDS=list(range(120000,120010))\nSEED_POLICY=\'stochastic: source25 full-data seeds120000..120009, cycled every10 blocks; deterministic: seed120000\'\nDEFAULT_CONFIG={\'benchmark_datasets\':None,\'timing_blocks\':30,\'warmup_repetitions\':1,\'force_timing\':False,\'force_audit\':False}\nSTAT_FILES=[\'01_dataset_values_and_ranks.csv\',\'02_average_ranks.csv\',\'03_friedman_kendall.csv\',\n    \'04_paired_wilcoxon_holm.csv\',\'05_paired_differences_and_ranks.csv\',\'status_statistik.json\']\nPREVIOUS_EIGHT_METHOD_CODE = {\'code\': {\'implementation_sha256\': \'8d4409262ffafc70d01dd6875e34ce6d4cd76dfaa93016240907ec6ebdaf5ab5\', \'helper25_sha256\': \'47227e8d9a8e150725e15d9895ff214a9c06ab92754653ae6e513f1c9055f784\', \'bk25_sha256\': \'4340c434adc901ad866c5c14a074804f17e97739cd13761953aa707cbaee0c56\', \'utils_sha256\': \'dd2ba3d4a2cb8bc9fbd11982c6243489ea2d80f58c0d587d35005c49fe374c0a\', \'notebook_code\': \'aaf55ffc98a629034c3a6c5832639d7d49a5619fc26b8efa495622f4a8bed713\'}, \'scientific_constants\': {\'STAGE\': \'31_Benchmark_Runtime_dan_Failure_Audit\', \'U25\': \'OUTPUT/TAMBAHAN/25_Perbandingan_Initializer_Terintegrasi\', \'C25\': \'OUTPUT/TAMBAHAN/KONTROL/25_Perbandingan_Initializer_Terintegrasi\', \'DETERMINISTIC\': [\'Cao\', \'Bai\', \'CARE-KM\', \'BK-Modes\'], \'STOCHASTIC\': [\'HR-2023\', \'Huang\', \'K-Modes++\', \'Random\'], \'TIMING_METHODS\': [\'Cao\', \'Bai\', \'CARE-KM\', \'BK-Modes\', \'Random\', \'Huang\', \'K-Modes++\', \'HR-2023\'], \'SOURCE_SEEDS\': [120000, 120001, 120002, 120003, 120004, 120005, 120006, 120007, 120008, 120009], \'SEED_POLICY\': \'stochastic: source25 full-data seeds120000..120009, cycled every10 blocks; deterministic: seed120000\', \'DEFAULT_CONFIG\': {\'benchmark_datasets\': None, \'timing_blocks\': 30, \'warmup_repetitions\': 1, \'force_timing\': False, \'force_audit\': False}, \'STAT_FILES\': [\'01_dataset_values_and_ranks.csv\', \'02_average_ranks.csv\', \'03_friedman_kendall.csv\', \'04_paired_wilcoxon_holm.csv\', \'05_paired_differences_and_ranks.csv\', \'status_statistik.json\']}, \'scientific_definitions\': {\'config_check\': \'9aaf3a87aedfd2f179a13759b22845bec9f98de5ac3b6f6648d67c6646041299\', \'utc\': \'9abf2ff85f74c22ff7426fdc388773e17615e53414aa3b56b04b5db9d1722e93\', \'timing_config\': \'97132ab3f557af5bd0853c9fbdd15a6adab40e00103ec8b6e6c5cd881322597c\', \'seed_for\': \'d841183838b4cebb03d4ce650948e94971c651de2c674c1ee0c78d48f3794f7b\', \'scientific_attempt\': \'43f796dad652efced1a20d4acd1ea3c135572c74527c926eefdd682b37a77481\', \'source_setup\': \'fc2ab6eaa389b26e23585110033dbda69411efeb0f16c28a3ca5ef82e2b72779\', \'historical_flags\': \'a1afae784893b12e2002726be798f0ec50b173da1c2f8a2e64562c498a505e0c\', \'summarize_history\': \'0f88923619eb8ec1fd30be92e628316d359d1ed70ebbd3e693c315a053fbcdeb\', \'audit_source\': \'14f097a4d2dc1f79e41b2b207fc70abe522e7b254ad3f18c199d3a1d0c70d3da\', \'benchmark_preflight\': \'312329f9788fdb9904d51444161fbff139c8992fbc75e62a334d79102a99c250\', \'hardware\': \'fe8a69342cb01cf04db2c063656349b7e59a077a83f5e4c5681047988c45615f\', \'execute_fit\': \'45beed9faa88fd4bbc5a2058b7151969ef2c59586aff2d5caaf17d995f743d82\', \'record_attempt\': \'2468b2d6bebdaaf877ed3ed61e73bc660a76255775b2297436e66f44eb3fe90c\', \'timing_dataset\': \'f714102fe15cb3128228614d080d3361e06a5a6714b34f5c167de67b94add92b\', \'runtime_summary\': \'5a8af491666aeef2f12f073f25021092c5537e47578b6b3a8290771a379cad33\', \'failure_summary\': \'436b8aaf9ab92d48e01d94fda64ac71265bf0ee575f2001490bff703ee2bf2cb\', \'figures\': \'0ad1d119a0349e3c0ea6c62aa97aba8f6013fb252eeb717bccdeec6f95de2b91\', \'archive_four_method_table\': \'df665c39dde37804bc9fd8125fb8efc1baa061a1216c034dc8ee114a217cefbc\'}, \'unchanged_utils_definitions\': {\'sha\': \'4ca54239024a9babf2c565292cf01da7feea57793089c16062c1fa24669cdfe6\', \'digest\': \'8995003a131cfa6f11b62612fc2f35d739b3692b25306d606920edbfa9b3ac10\', \'clean\': \'75f35294d88db05471d19923c034486d9b5c16a5da748a8415b2c123d91f6a28\', \'atomic\': \'c9425ed5ee1eb60ce8acd8ca2e5874949ef46f6fec94530c086897fe90d29538\', \'save_json\': \'b9e6f09b001947d201940e03582fcdf3c7e5ea45de87769500bba1e6ffd7d072\', \'save_csv\': \'a5c13d1b50b8406adfa671bb3ead01a876b344e580313c22da9d03f25acc3a3b\', \'read_checkpoint\': \'329cf39cf2d9986629fd253e3941cc12ff5eba7a10a67b59edcf69db48821748\', \'Files.__init__\': \'ffbcb1cffc1c92a78881efc216f57a226b73bdb89c2d97894f192251ba41c753\', \'Files.path\': \'9913614b1427f3f771c29406df0efcde2b8669a50f396eca10a2cf720df45acf\', \'Files.hash\': \'9a538837fef860b9ac85641dce81da5cedd7a0ced2bb78e8709ed00a155033d7\', \'Files.frame\': \'885ab895c032507cc779f93ead391a69e829690f88860c6a13838d9c5fbe1d61\', \'runtime\': \'42c39e1d0be6e11d93e6140628f5527a4bb42fdcb2d96f104dee27b77d710469\', \'notebook_hash\': \'cfae0ce8282a4f90bab5860509edc24f74dd31442254e058d6dd8d54300af85e\', \'cache_read\': \'5ac82411cb0e05afe67778ec7d095dea9a0c5b0b600dcab8dc2af92cca2b377a\', \'cache_write\': \'7b06d4366d08dc7ab2b8ee9f32b3e9d9fb5857f0b6843f98628aae4161088dcd\', \'unchanged\': \'fd541bb3b7f3b0d75284e32b63631145cb04897840c79745958d9500b9984199\'}}\n\n\ndef config_check(config,order):\n    if set(config)-set(DEFAULT_CONFIG):raise ValueError(\'Konfigurasi tidak dikenal.\')\n    c=dict(DEFAULT_CONFIG);c.update(config);c[\'benchmark_datasets\']=list(order if c[\'benchmark_datasets\'] is None else c[\'benchmark_datasets\'])\n    if not c[\'benchmark_datasets\'] or len(set(c[\'benchmark_datasets\']))!=len(c[\'benchmark_datasets\']) or set(c[\'benchmark_datasets\'])-set(order):raise ValueError(\'Pilihan benchmark tidak sesuai registry.\')\n    for key in [\'timing_blocks\',\'warmup_repetitions\']:\n        if isinstance(c[key],bool) or not isinstance(c[key],int) or c[key]<1:raise ValueError(key+\' harus integer positif.\')\n    for key in [\'force_timing\',\'force_audit\']:\n        if not isinstance(c[key],bool):raise ValueError(key+\' harus True/False.\')\n    return c\n\n\ndef utc():return datetime.now(timezone.utc).isoformat()\n\n\ndef timing_config(cfg):return {k:v for k,v in cfg.items() if not k.startswith(\'force_\')}\n\n\ndef seed_for(method,block):\n    source_run=block%len(SOURCE_SEEDS) if method in STOCHASTIC else 0\n    return SOURCE_SEEDS[source_run],source_run\n\n\ndef scientific_attempt(ctx,method,block,data,warmup):\n    seed,source_run=seed_for(method,block)\n    return {\'dataset\':ctx[\'dataset\'],\'method\':method,\'block\':block,\'matrix_sha256\':ctx[\'matrix_sha256\'],\n        \'k\':ctx[\'k\'],\'max_iter\':100,\'care_config\':data[\'constants\'][\'FROZEN_CARE_CONFIG\'],\'seed\':seed,\n        \'source_run\':source_run,\'seed_policy\':SEED_POLICY,\'warmup\':warmup}\n\n\ndef source_setup(u,h,files,helper_source,bk_source):\n    protocol_path=U25+\'/PROTOKOL/02_konfigurasi.json\';p=json.loads(files.read(protocol_path))\n    status=json.loads(files.read(C25+\'/status_eksekusi.json\'))\n    if not status.get(\'full_protocol\') or status.get(\'completed_jobs\')!=3272 or status.get(\'status\') not in [\'selesai\',\'selesai_dengan_kegagalan_algoritmik\']:\n        raise RuntimeError(\'Tahap 25 harus selesai dengan cakupan penuh sebelum tahap31.\')\n    if p[\'constants\']!={\'FROZEN_CARE_CONFIG\':{\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0},\'K\':16,\'MAX_ITER\':100,\'SI_SAMPLE_SIZE\':2000}:\n        raise RuntimeError(\'Konfigurasi sumber bukan reference yang didukung.\')\n    if h.notebook_code(files)!=p[\'code\'][\'notebook_code\']:raise RuntimeError(\'Kode notebook25 berubah setelah hasil dibuat.\')\n    nb=json.loads(files.read(\'PYTHON/\'+h.STAGE+\'.ipynb\'));literals={}\n    for c in nb[\'cells\']:\n        if c[\'cell_type\']==\'code\':\n            for n in ast.parse(\'\'.join(c[\'source\'])).body:\n                if isinstance(n,ast.Assign) and len(n.targets)==1 and isinstance(n.targets[0],ast.Name) and isinstance(n.value,ast.Constant):literals[n.targets[0].id]=n.value.value\n    for key,name in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if u.sha(literals[name].encode())!=p[\'code\'][\'embedded_sources\'][key]:raise RuntimeError(\'Sumber tertanam25 berbeda: \'+key)\n    if u.sha(helper_source.encode())!=p[\'code\'][\'embedded_sources\'][\'comparison\'] or u.sha(bk_source.encode())!=p[\'code\'][\'embedded_sources\'][\'initializer\']:\n        raise RuntimeError(\'Helper/initializer beku tidak sesuai hasil25.\')\n    watched={protocol_path:files.sha(protocol_path)}\n    case_path=U25+\'/VERIFIKASI/01_kasus_kecil.csv\';case_record_path=C25+\'/verifikasi_kecil.json\'\n    cases=files.frame(case_path);case_record=json.loads(files.read(case_record_path))\n    if len(cases)!=55 or not cases.status.eq(\'LULUS\').all() or case_record[\'key\']!=h.digest({\'code\':p[\'code\'],\'runtime\':p[\'runtime\']}) or case_record[\'output_sha256\']!=files.sha(case_path):\n        raise RuntimeError(\'Verifikasi initializer55 pemeriksaan sumber tidak sesuai.\')\n    watched.update({case_path:files.sha(case_path),case_record_path:files.sha(case_record_path)})\n    for path,want in p[\'code\'][\'project_libraries\'].items():\n        if files.sha(path)!=want:raise RuntimeError(\'Pustaka ilmiah berubah: \'+path)\n        watched[path]=want\n    core,gd,constants,_=h.load_project(files)\n    if set(TIMING_METHODS)!=set(h.METHODS) or STOCHASTIC!=h.STOCHASTIC:raise RuntimeError(\'Daftar metode/kelompok stokastik berbeda dari sumber25.\')\n    if p[\'config\'][\'benchmark_datasets\']!=gd.DATASET_ORDER:raise RuntimeError(\'Daftar sepuluh dataset sumber berbeda.\')\n    rawpath=U25+\'/01_hasil_per_run_8_metode.csv\';summarypath=U25+\'/02_ringkasan_8_metode.csv\'\n    raw=files.frame(rawpath);summary=files.frame(summarypath)\n    if len(raw)!=3272 or set(raw.method)!=set(h.METHODS):raise RuntimeError(\'Tabel kualitas25 belum lengkap.\')\n    for path in [rawpath,summarypath,U25+\'/04_failure_audit.csv\']+[U25+\'/STATISTIK_BENCHMARK/\'+f for f in STAT_FILES]:watched[path]=files.sha(path)\n    checks=[(h.STAGE,\'initializer25_configuration\',\'KONFIGURASI_25\',p[\'code\'][\'notebook_code\'])]\n    return {\'protocol\':p,\'core\':core,\'gd\':gd,\'constants\':constants,\'raw\':raw,\'summary\':summary,\'watched\':watched,\'checks\':checks}\n\n\ndef historical_flags(row):\n    def finite(key):\n        try:return np.isfinite(float(row[key]))\n        except (TypeError,ValueError,KeyError):return False\n    invalid_metric=any(not finite(k) for k in [\'dbi\',\'si\'])\n    invalid_structure_numeric=any(not finite(k) for k in [\'initial_cost\',\'final_cost\',\'iterations\',\'min_cluster\',\'max_cluster\',\'cluster_cv\'])\n    empty=finite(\'min_cluster\') and float(row[\'min_cluster\'])<=0\n    worse=finite(\'initial_cost\') and finite(\'final_cost\') and float(row[\'final_cost\'])>float(row[\'initial_cost\'])+1e-12\n    baditer=finite(\'iterations\') and float(row[\'iterations\'])<=0\n    return {\'invalid_numeric\':invalid_structure_numeric,\'metric_undefined\':invalid_metric,\'empty_final_cluster\':empty,\n        \'objective_worsened\':worse,\'nonpositive_iterations\':baditer,\'iteration_cap_reached\':finite(\'iterations\') and int(row[\'iterations\'])>=100,\n        \'cluster_cv_gt_1\':finite(\'cluster_cv\') and float(row[\'cluster_cv\'])>1,\n        \'source_invalid\':row.get(\'valid\') is not True,\'algorithmic_failure\':invalid_structure_numeric or invalid_metric or empty or worse or baditer or row.get(\'valid\') is not True}\n\n\ndef summarize_history(frame,by_dataset=True):\n    columns=[\'scope\']+([\'dataset\'] if by_dataset else [])+[\'method\'];rows=[]\n    flags=[\'invalid_numeric\',\'metric_undefined\',\'empty_final_cluster\',\'objective_worsened\',\'nonpositive_iterations\',\'iteration_cap_reached\',\'cluster_cv_gt_1\',\'source_invalid\',\'algorithmic_failure\']\n    for key,g in frame.groupby(columns,sort=False):\n        row=dict(zip(columns,key));row.update(n_runs=len(g),n_success=int((~g.algorithmic_failure).sum()),n_failures=int(g.algorithmic_failure.sum()))\n        row[\'failure_rate\']=row[\'n_failures\']/len(g)\n        for f in flags:row[f+\'_runs\']=int(g[f].sum())\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef audit_source(u,h,files,out,control,data,code,cfg):\n    p=data[\'protocol\'];raw=data[\'raw\'];watched=dict(data[\'watched\']);canonical=[];flags=[];seen=set();record_paths=[]\n    for i,r in enumerate(raw.to_dict(\'records\')):\n        identity={\'scope\':r[\'scope\'],\'dataset\':r[\'dataset\'],\'rep\':None if pd.isna(r[\'rep\']) else int(r[\'rep\']),\'run\':int(r[\'run\']),\'method\':r[\'method\']}\n        key=h.digest(identity)[:24];path=C25+\'/JOBS/\'+key+\'.json\';record_paths.append(path)\n        if key in seen:raise RuntimeError(\'Identitas run sumber berulang.\')\n        seen.add(key);rec=json.loads(files.read(path));want=rec.pop(\'record_sha256\',\'\')\n        if h.digest(rec)!=want or rec[\'identity\']!=identity or rec[\'status\']!=\'selesai\' or rec[\'signature\']!=h.digest({\'parameters\':rec[\'parameters\'],\'code\':p[\'code\'],\'runtime\':p[\'runtime\']}):\n            raise RuntimeError(\'Catatan sumber25 tidak sah: \'+path)\n        original=rec[\'row\'];watched[path]=files.sha(path)\n        for field in [\'scope\',\'dataset\',\'method\',\'run\',\'seed\',\'k\',\'sample_n\',\'matrix_sha256\',\'origin\',\'valid\',\'initial_cost\',\'final_cost\',\'dbi\',\'si\',\'iterations\',\'min_cluster\',\'max_cluster\',\'cluster_cv\']:\n            a=r.get(field);b=original.get(field)\n            if pd.isna(a) and b is None:continue\n            if a!=b:raise RuntimeError(\'Tabel agregat dan catatan sumber berbeda pada \'+field+\': \'+path)\n        for artifact,checksum in rec[\'outputs\'].items():\n            if files.sha(artifact)!=checksum:raise RuntimeError(\'Artefak sumber berubah: \'+artifact)\n            watched[artifact]=checksum\n        ev=rec.get(\'legacy_evidence\')\n        if ev:\n            if files.sha(ev[\'file\'])!=ev[\'sha256\']:raise RuntimeError(\'Bukti reuse berubah: \'+ev[\'file\'])\n            watched[ev[\'file\']]=ev[\'sha256\']\n        canonical.append(original);flags.append({**identity,\'n\':original[\'sample_n\'],\'d\':original[\'features\'],\'k\':original[\'k\'],\n            \'origin\':original[\'origin\'],\'source_failure_flags\':original.get(\'failure_flags\',\'\'),\'source_record\':path,**historical_flags(original)})\n        if (i+1)%500==0:print(\'Audit sumber:\',i+1,\'/\',len(raw),flush=True)\n    source=pd.DataFrame(canonical);audit=pd.DataFrame(flags);benchmark=audit.loc[audit.scope.str.startswith(\'BENCHMARK\')].copy()\n    if len(benchmark)!=2840 or len(benchmark.loc[benchmark.method.eq(\'BK-Modes\')])!=310:raise RuntimeError(\'Cakupan benchmark bukan2840/BK310.\')\n    expected=h.planned_jobs(p[\'config\'])\n    if source.groupby(\'scope\').size().to_dict()!=expected:raise RuntimeError(\'Cakupan kelompok sumber berbeda dari protokol.\')\n    for (scope,ds,method),g in source.groupby([\'scope\',\'dataset\',\'method\']):\n        if scope in [\'BENCHMARK_FULL\',\'BENCHMARK_RESAMPLING\']:\n            count=30 if scope.endswith(\'RESAMPLING\') else 10 if method in h.STOCHASTIC else 1\n            if len(g)!=count or (scope.endswith(\'RESAMPLING\') and sorted(g.rep.astype(int))!=list(range(30))):raise RuntimeError(\'Jumlah benchmark run/rep tidak sesuai.\')\n    calculated=h.summarize(source);stored=data[\'summary\']\n    merged=stored.merge(calculated,on=[\'scope\',\'dataset\',\'method\'],suffixes=(\'_stored\',\'_calculated\'),validate=\'one_to_one\')\n    if len(merged)!=len(stored) or len(merged)!=len(calculated):raise RuntimeError(\'Ringkasan sumber tidak lengkap.\')\n    for col in [\'n_runs\',\'n_valid\',\'n_failures\',\'partition_runs\']+[m+x for m in h.METRICS for x in [\'_mean\',\'_sd\']]:\n        if not np.allclose(merged[col+\'_stored\'],merged[col+\'_calculated\'],rtol=1e-12,atol=1e-12,equal_nan=True):raise RuntimeError(\'Ringkasan sumber berubah: \'+col)\n    rank=files.frame(U25+\'/STATISTIK_BENCHMARK/01_dataset_values_and_ranks.csv\')\n    for (scope,metric),g in rank.groupby([\'source\',\'metric\']):\n        q=stored.loc[stored.scope.eq(scope)].pivot(index=\'dataset\',columns=\'method\',values=metric+\'_mean\').reindex(index=data[\'gd\'].DATASET_ORDER,columns=h.METHODS)\n        rr=q.rank(axis=1,ascending=metric!=\'si\',method=\'average\')\n        if len(g)!=80:raise RuntimeError(\'Nilai/rank tidak lengkap.\')\n        for r in g.itertuples():\n            if not np.isclose(r.value,q.loc[r.dataset,r.method],rtol=1e-12,atol=1e-12) or r.rank!=rr.loc[r.dataset,r.method]:raise RuntimeError(\'Rank sumber tidak sesuai kualitas.\')\n    signature=u.digest({\'code\':code,\'sources\':watched});cp=control/\'audit_historis.json\';cached=u.cache_read(files,cp,signature,cfg[\'force_audit\'])\n    if not cached:\n        paths=[]\n        for name,frame in [(\'AUDIT_HISTORIS/01_per_run_benchmark.csv\',benchmark),(\'AUDIT_HISTORIS/02_per_dataset_metode.csv\',summarize_history(benchmark)),\n            (\'AUDIT_HISTORIS/03_per_sumber_metode.csv\',summarize_history(benchmark,False))]:\n            path=out/name;u.save_csv(path,frame);paths.append(path)\n        for name in [\'01_hasil_per_run_8_metode.csv\',\'02_ringkasan_8_metode.csv\']:\n            path=out/\'KUALITAS\'/name;u.atomic(path,files.read(U25+\'/\'+name));paths.append(path)\n        path=out/\'KUALITAS/VERIFIKASI/01_kasus_kecil.csv\';u.atomic(path,files.read(U25+\'/VERIFIKASI/01_kasus_kecil.csv\'));paths.append(path)\n        for name in STAT_FILES:\n            path=out/\'KUALITAS/STATISTIK_BENCHMARK\'/name;u.atomic(path,files.read(U25+\'/STATISTIK_BENCHMARK/\'+name));paths.append(path)\n        for scope,name in [(\'LMS_FULL\',\'01_lms_full_8_metode.csv\'),(\'LMS_RESAMPLING\',\'02_lms_resampling_8_metode.csv\'),\n            (\'BENCHMARK_FULL\',\'03_benchmark_full_8_metode.csv\'),(\'BENCHMARK_RESAMPLING\',\'04_benchmark_resampling_8_metode.csv\')]:\n            path=out/\'TABEL\'/name;u.save_csv(path,stored.loc[stored.scope.eq(scope)]);paths.append(path)\n        path=out/\'TABEL/05_average_ranks_8_metode.csv\';u.atomic(path,files.read(U25+\'/STATISTIK_BENCHMARK/02_average_ranks.csv\'));paths.append(path)\n        path=out/\'TABEL/06_failure_historis_8_metode.csv\';u.save_csv(path,summarize_history(benchmark,False));paths.append(path)\n        u.cache_write(files,cp,signature,paths,{\'verified_source_runs\':len(source),\'benchmark_runs\':len(benchmark)})\n    print(\'Kualitas/audit:3272 catatan terverifikasi;2840 benchmark,310 BK-Modes.\',flush=True)\n    data.update(source=source,historical=benchmark,watched=watched,historical_cached=cached is not None)\n    return data\n\n\ndef benchmark_preflight(u,files,data,cfg,out):\n    gd=data[\'gd\'];p=data[\'protocol\'];missing=[];problems=[];inputs={};rows=[]\n    for ds in cfg[\'benchmark_datasets\']:\n        path=\'DATA/\'+gd.DATASETS[ds][\'file\'];want=p[\'input_hashes\'].get(path)\n        present=files.path(path).is_file();actual=files.sha(path) if present else None\n        rows.append({\'dataset\':ds,\'path\':path,\'present\':present,\'expected_sha256\':want,\'actual_sha256\':actual,\'matches_source\':present and actual==want})\n        if not present:missing.append(path)\n        elif not want or actual!=want:problems.append(path)\n        else:inputs[path]=actual\n    u.save_csv(out/\'PROTOKOL/04_pemeriksaan_DATA.csv\',pd.DataFrame(rows))\n    if missing:raise FileNotFoundError(\'Arsip benchmark belum tersedia di MyDrive/REVISED/DATA. Hasil audit/kualitas sudah disimpan. Lengkapi berkas berikut tanpa mengganti isinya:\\n\'+\'\\n\'.join(missing))\n    if problems:raise RuntimeError(\'Isi arsip DATA berbeda dari sumber25; jangan melanjutkan timing dengan data berbeda:\\n\'+\'\\n\'.join(problems))\n    for name in [\'chronic_kidney_disease_full.arff\',\'chronic_kidney_disease.arff\']:\n        path=\'DATA/\'+name\n        if \'chronic_kidney\' in cfg[\'benchmark_datasets\'] and files.path(path).is_file():\n            actual=files.sha(path);want=p[\'input_hashes\'].get(path)\n            if want and want!=actual:raise RuntimeError(\'ARFF fallback berubah: \'+path)\n            inputs[path]=actual\n    data[\'watched\'].update(inputs)\n\n\ndef hardware(u):\n    from threadpoolctl import threadpool_info\n    import psutil\n    cpu=platform.processor()\n    if Path(\'/proc/cpuinfo\').is_file():cpu=next((line.split(\':\',1)[1].strip() for line in Path(\'/proc/cpuinfo\').read_text().splitlines() if line.startswith(\'model name\')),cpu)\n    boot=Path(\'/proc/sys/kernel/random/boot_id\')\n    machine={\'node\':platform.node(),\'system\':platform.system(),\'release\':platform.release(),\'architecture\':platform.machine(),\'CPU\':cpu,\n        \'logical_cpus\':os.cpu_count(),\'RAM_bytes\':int(psutil.virtual_memory().total),\'boot_identity\':boot.read_text().strip() if boot.is_file() else str(int(psutil.boot_time()))}\n    pools=[{k:x.get(k) for k in [\'user_api\',\'internal_api\',\'prefix\',\'version\',\'threading_layer\',\'architecture\',\'num_threads\']} for x in threadpool_info()]\n    return {\'machine\':machine,\'threadpools\':pools,\'thread_limit\':1,\'identity_sha256\':u.digest({\'machine\':machine,\'threadpools\':pools,\'thread_limit\':1}),\n        \'timer\':\'perf_counter_ns\',\'timer_resolution_seconds\':time.get_clock_info(\'perf_counter\').resolution}\n\n\ndef execute_fit(u,core,bk,ctx,method,constants,identity):\n    """All initializer/optimizer work is timed; audit and fingerprints are outside."""\n    X=ctx[\'X\'];k=ctx[\'k\'];row={**identity,\'n\':len(X),\'d\':X.shape[1],\'k\':k,\n        \'init_sec\':None,\'kmodes_sec\':None,\'total_sec\':None,\'elapsed_to_failure_sec\':None,\n        \'outcome\':\'RUNNING\',\'exception_type\':\'\',\'exception_message\':\'\',\'fit_valid\':False,\'initial_cost\':None,\'final_cost\':None,\'iterations\':None,\n        \'empty_final_cluster\':False,\'objective_worsened\':False,\'nonpositive_iterations\':False,\'invalid_numeric\':False,\n        \'iteration_cap_reached\':False,\'cluster_cv_gt_1\':False,\'metric_evaluation\':\'not timed/not recomputed\',\n        \'reference_match\':None,\'reproducibility_flags\':\'\',\'started_utc\':utc()}\n    phase=\'INITIALIZER\';t0=time.perf_counter_ns();t1=None;detail=None\n    try:\n        if method==\'BK-Modes\':\n            detail=bk.bk_modes_init(X,k,max_iter=constants[\'MAX_ITER\'],return_details=True);init=detail[\'modes\']\n        else:init=core.init_by_name(method,X,k,seed=identity[\'seed\'],care_config=constants[\'FROZEN_CARE_CONFIG\'])\n        t1=time.perf_counter_ns();phase=\'KModes\'\n        fit=core.kmodes_fit(X,init,max_iter=constants[\'MAX_ITER\']);t2=time.perf_counter_ns()\n        row.update(init_sec=(t1-t0)/1e9,kmodes_sec=(t2-t1)/1e9,total_sec=(t2-t0)/1e9)\n        phase=\'AUDIT\';labels=np.asarray(fit[\'labels\']);modes=np.asarray(fit[\'modes\'],object);init=np.asarray(init,object)\n        if init.shape!=(k,X.shape[1]) or modes.shape!=init.shape or labels.shape!=(len(X),) or not np.isin(labels,np.arange(k)).all():raise ValueError(\'Bentuk/label mode hasil tidak sah.\')\n        labels=labels.astype(np.int64);counts=np.bincount(labels,minlength=k)\n        initial=float(core.initial_cost(X,init));final=float(fit[\'cost\']);iterations=int(fit[\'iterations\'])\n        closest,distance=core.assign_nearest(X,modes)\n        flags=[]\n        if not np.array_equal(labels,closest) or final!=float(distance.sum()):flags.append(\'cost_or_assignment_inconsistent\')\n        if any(not np.isin(modes[:,j],np.unique(X[:,j])).all() for j in range(X.shape[1])):flags.append(\'unknown_mode_category\')\n        empty=bool((counts==0).any());worse=bool(final>initial+1e-12);baditer=iterations<1\n        invalid=not np.isfinite([initial,final,iterations,row[\'init_sec\'],row[\'kmodes_sec\'],row[\'total_sec\']]).all()\n        if empty:flags.append(\'empty_final_cluster\')\n        if worse:flags.append(\'objective_worsened\')\n        if baditer:flags.append(\'nonpositive_iterations\')\n        if invalid:flags.append(\'invalid_numeric\')\n        cv=float(counts.std(ddof=0)/counts.mean())\n        row.update(initial_cost=initial,final_cost=final,iterations=iterations,min_cluster=int(counts.min()),max_cluster=int(counts.max()),cluster_cv=cv,\n            empty_final_cluster=empty,objective_worsened=worse,nonpositive_iterations=baditer,invalid_numeric=bool(invalid),\n            iteration_cap_reached=iterations>=constants[\'MAX_ITER\'],cluster_cv_gt_1=cv>1,fit_valid=not flags,\n            outcome=\'OK\' if not flags else \'FIT_INVALID\',failure_flags=\'; \'.join(flags),\n            initial_modes_sha256=u.digest(init.tolist()),final_modes_sha256=u.digest(modes.tolist()),labels_sha256=u.sha(labels.tobytes()),\n            unique_initial_modes=len({tuple(x) for x in init}),bk_inner_iterations=detail[\'total_two_modes_iterations\'] if detail else None,\n            bk_inner_empty_repairs=detail[\'empty_repairs\'] if detail else None)\n        if row[\'fit_valid\']:\n            reference=ctx[\'reference\'][method][identity[\'source_run\']];differences=[]\n            if int(reference[\'seed\'])!=identity[\'seed\']:differences.append(\'source_seed\')\n            for field in [\'initial_cost\',\'final_cost\',\'iterations\']:\n                if reference.get(field) is not None and float(reference[field])!=row[field]:differences.append(field)\n            row[\'reference_match\']=not differences;row[\'reproducibility_flags\']=\'; \'.join(differences)\n        row[\'finished_utc\']=utc();return row,{\'init\':init,\'fit\':fit,\'detail\':detail}\n    except Exception as error:\n        elapsed=(time.perf_counter_ns()-t0)/1e9\n        if t1 is not None and row[\'init_sec\'] is None:row[\'init_sec\']=(t1-t0)/1e9\n        row.update(outcome=\'RESOURCE_ERROR\' if isinstance(error,MemoryError) else phase+\'_ERROR\',exception_type=type(error).__name__,\n            exception_message=str(error),elapsed_to_failure_sec=elapsed,failure_flags=phase+\'_exception\',finished_utc=utc())\n        return row,None\n\n\ndef record_attempt(u,files,out,campaign_control,ctx,method,block,position,data,code,cfg,hw,bk,run_id,warmup=False):\n    seed,source_run=seed_for(method,block)\n    identity={\'dataset\':ctx[\'dataset\'],\'method\':method,\'block\':block,\'order_position\':position,\'phase\':\'WARMUP\' if warmup else \'TIMED\',\n        \'hardware_sha256\':hw[\'identity_sha256\'],\'matrix_sha256\':ctx[\'matrix_sha256\'],\'run_id\':run_id,\n        \'seed\':seed,\'source_run\':source_run,\'method_type\':\'stochastic\' if method in STOCHASTIC else \'deterministic\'}\n    scientific=scientific_attempt(ctx,method,block,data,warmup)\n    sig=u.digest({\'code\':code,\'hardware\':hw,\'scientific\':scientific,\'configuration\':timing_config(cfg)})\n    stem=method.replace(\'++\',\'_PP\').replace(\'-\',\'_\');cp=campaign_control/(\'WARMUP/\'+run_id if warmup else \'ATTEMPTS\')/ctx[\'dataset\']/(\'%03d\'%block)/(stem+\'.json\')\n    if not warmup:\n        cached=u.cache_read(files,cp,sig,cfg[\'force_timing\'])\n        if cached:\n            if cached[\'row\'].get(\'phase\')!=\'TIMED\' or cached[\'row\'].get(\'hardware_sha256\')!=hw[\'identity_sha256\']:raise RuntimeError(\'Identitas cache timing berbeda.\')\n            return cached[\'row\'],\'CACHE\'\n    old=u.read_checkpoint(cp)\n    if old and old.get(\'status\') in [\'berjalan\',\'terinterupsi\']:\n        u.save_json(campaign_control/\'INTERRUPTIONS\'/(uuid.uuid4().hex+\'.json\'),old)\n    elif old and old.get(\'status\')==\'selesai\':\n        previous=dict(old);checksum=previous.pop(\'record_sha256\',\'\')\n        valid_previous=bool(checksum and u.digest(previous)==checksum)\n        u.save_json(campaign_control/\'REPLACED_ATTEMPTS\'/(uuid.uuid4().hex+\'.json\'),{\n            \'reason\':\'explicit_force\' if cfg[\'force_timing\'] else \'invalidated_cached_artifact\',\n            \'archived_utc\':utc(),\'previous_record_checksum_valid\':valid_previous,\'previous_record\':old})\n    u.save_json(cp,{\'status\':\'berjalan\',\'signature\':sig,\'identity\':identity,\'started_utc\':utc()})\n    try:\n        row,payload=execute_fit(u,data[\'core\'],bk,ctx,method,data[\'constants\'],identity)\n        outputs=[]\n        if not warmup and block==0 and payload is not None:\n            folder=out/\'PARTISI\'/campaign_control.name/ctx[\'dataset\']/stem\n            for name,array in [(\'01_initial_modes.csv\',payload[\'init\']),(\'02_final_modes.csv\',payload[\'fit\'][\'modes\'])]:\n                frame=pd.DataFrame(array,columns=ctx[\'features\']);frame.insert(0,\'cluster_id\',np.arange(ctx[\'k\']));path=folder/name;u.save_csv(path,frame);outputs.append(path)\n            path=folder/\'03_assignments.csv.gz\';u.save_csv(path,pd.DataFrame({\'source_row_index\':np.arange(len(ctx[\'X\'])),\'cluster_id\':payload[\'fit\'][\'labels\']}));outputs.append(path)\n            if payload[\'detail\'] is not None:path=folder/\'04_bisection_trace.json\';u.save_json(path,payload[\'detail\'][\'split_trace\']);outputs.append(path)\n        u.cache_write(files,cp,sig,outputs,{\'row\':row,\'scientific\':scientific})\n        return row,\'MEASURE\'\n    except BaseException as error:\n        u.save_json(cp,{\'status\':\'terinterupsi\',\'signature\':sig,\'identity\':identity,\'error\':type(error).__name__+\': \'+str(error),\'finished_utc\':utc()})\n        raise\n\n\ndef timing_dataset(u,h,files,out,campaign_control,ctx,data,code,cfg,hw,bk,run_id):\n    ds=ctx[\'dataset\'];rows=[];warmups=[];counts={\'MEASURE\':0,\'CACHE\':0};warmed=False\n    for block in range(cfg[\'timing_blocks\']):\n        offset=block%len(TIMING_METHODS);order=TIMING_METHODS[offset:]+TIMING_METHODS[:offset]\n        for position,method in enumerate(order):\n            if not warmed:\n                # Avoid running warm-up when all requested attempts already have valid cache.\n                pending=False\n                for b in range(cfg[\'timing_blocks\']):\n                    offset=b%len(TIMING_METHODS)\n                    for pos,m in enumerate(TIMING_METHODS[offset:]+TIMING_METHODS[:offset]):\n                        science=scientific_attempt(ctx,m,b,data,False)\n                        sig=u.digest({\'code\':code,\'hardware\':hw,\'scientific\':science,\'configuration\':timing_config(cfg)})\n                        cp=campaign_control/\'ATTEMPTS\'/ds/(\'%03d\'%b)/(m.replace(\'++\',\'_PP\').replace(\'-\',\'_\')+\'.json\')\n                        if u.cache_read(files,cp,sig,cfg[\'force_timing\']) is None:pending=True;break\n                    if pending:break\n                if pending:\n                    for wi in range(cfg[\'warmup_repetitions\']):\n                        for pos,m in enumerate(TIMING_METHODS):\n                            r,_=record_attempt(u,files,out,campaign_control,ctx,m,wi,pos,data,code,cfg,hw,bk,run_id,True);warmups.append(r)\n                warmed=True\n            r,origin=record_attempt(u,files,out,campaign_control,ctx,method,block,position,data,code,cfg,hw,bk,run_id)\n            rows.append(r);counts[origin]+=1\n        print(\'Runtime\',ds,\'blok\',block+1,\'/\',cfg[\'timing_blocks\'],counts,flush=True)\n        u.save_csv(out/\'RUNTIME/PER_DATASET\'/(ds+\'.csv\'),pd.DataFrame(rows))\n    return rows,warmups,counts\n\n\ndef runtime_summary(frame,by_seed=False):\n    rows=[]\n    keys=[\'dataset\',\'method\']+([\'seed\'] if by_seed else [])\n    for group,g in frame.groupby(keys,sort=False):\n        ds,method=group[:2]\n        good=g.loc[g.outcome.eq(\'OK\')];row={\'dataset\':ds,\'method\':method,\'n\':int(g.n.iloc[0]),\'d\':int(g.d.iloc[0]),\'k\':int(g.k.iloc[0]),\n            \'n_attempted\':len(g),\'n_success\':len(good),\'n_failure\':len(g)-len(good),\'failure_rate\':1-len(good)/len(g),\n            \'n_reference_mismatch\':int(g.reference_match.eq(False).sum()),\'n_iteration_cap\':int(g.iteration_cap_reached.eq(True).sum())}\n        for key in [\'init_sec\',\'kmodes_sec\',\'total_sec\']:\n            values=pd.to_numeric(good[key],errors=\'coerce\');row.update({key+\'_median\':values.median(),key+\'_mean\':values.mean(),key+\'_sd\':values.std(ddof=1),\n                key+\'_p25\':values.quantile(.25),key+\'_p75\':values.quantile(.75),key+\'_p95\':values.quantile(.95)})\n        fingerprints=[\'initial_modes_sha256\',\'final_modes_sha256\',\'labels_sha256\',\'initial_cost\',\'final_cost\',\'iterations\']\n        checks=[all(part[k].nunique(dropna=False)<=1 for k in fingerprints if k in part) for _,part in good.groupby(\'seed\')]\n        consistent=all(checks) if checks else None\n        row.update(method_type=\'stochastic\' if method in STOCHASTIC else \'deterministic\',n_unique_seeds=int(g.seed.nunique()),\n            source_runs=json.dumps(sorted(g.source_run.unique().astype(int).tolist())),within_seed_repeats_consistent=consistent,\n            deterministic_repeats_consistent=consistent if method in DETERMINISTIC else None,\n            n_seed_groups_with_repeats=int((good.groupby(\'seed\').size()>1).sum()),\n            n_distinct_initial_mode_sets=int(good.initial_modes_sha256.nunique()) if \'initial_modes_sha256\' in good else 0)\n        if by_seed:row[\'seed\']=int(group[2]);row[\'source_run\']=int(g.source_run.iloc[0])\n        rows.append(row)\n    table=pd.DataFrame(rows)\n    for ds,idx in table.groupby(\'dataset\').groups.items():\n        cao=frame.loc[frame.dataset.eq(ds)&frame.method.eq(\'Cao\')&frame.outcome.eq(\'OK\')]\n        for key in [\'init_sec\',\'kmodes_sec\',\'total_sec\']:table.loc[idx,key+\'_ratio_to_Cao\']=table.loc[idx,key+\'_median\']/pd.to_numeric(cao[key],errors=\'coerce\').median()\n    return table\n\n\ndef failure_summary(frame):\n    rows=[]\n    for (ds,method),g in frame.groupby([\'dataset\',\'method\'],sort=False):\n        row={\'dataset\':ds,\'method\':method,\'method_type\':\'stochastic\' if method in STOCHASTIC else \'deterministic\',\n            \'n_unique_seeds\':int(g.seed.nunique()),\'n_attempted\':len(g),\'n_success\':int(g.outcome.eq(\'OK\').sum()),\'n_failed\':int(g.outcome.ne(\'OK\').sum())}\n        for key in [\'empty_final_cluster\',\'objective_worsened\',\'nonpositive_iterations\',\'invalid_numeric\',\'iteration_cap_reached\',\'cluster_cv_gt_1\']:\n            row[key+\'_runs\']=int(g[key].eq(True).sum())\n        row.update(resource_error_runs=int(g.outcome.eq(\'RESOURCE_ERROR\').sum()),initializer_error_runs=int(g.outcome.eq(\'INITIALIZER_ERROR\').sum()),\n            optimizer_error_runs=int(g.outcome.eq(\'KModes_ERROR\').sum()),audit_error_runs=int(g.outcome.eq(\'AUDIT_ERROR\').sum()),\n            reference_mismatch_runs=int(g.reference_match.eq(False).sum()),failure_rate=row[\'n_failed\']/len(g))\n        elapsed=pd.to_numeric(g.loc[g.outcome.ne(\'OK\'),\'elapsed_to_failure_sec\'],errors=\'coerce\');row[\'elapsed_to_failure_sec_median\']=elapsed.median();rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef figures(u,out,summary,datasets):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    labels={\'car\':\'Car\',\'lymphography\':\'Lymph.\',\'hayes_roth\':\'Hayes\',\'australian_credit\':\'Aus.\',\n        \'chronic_kidney\':\'CKD\',\'annealing\':\'Anneal.\',\'statlog_heart\':\'S.Heart\',\'heart_disease\':\'Cleveland\',\'thyroid_disease\':\'Thyroid\',\'nursery\':\'Nursery\'}\n    colors=[\'#647b8c\',\'#c18b2b\',\'#185785\',\'#9451a6\',\'#dc7545\',\'#2b8c67\',\'#bd414d\',\'#168e9c\'];paths=[]\n    for number,key,title in [(\'01\',\'init_sec\',\'Initialization\'),(\'02\',\'total_sec\',\'Initialization plus final K-Modes\')]:\n        fig,ax=plt.subplots(figsize=(8.3,5.5));positive=False\n        for mi,(method,color) in enumerate(zip(TIMING_METHODS,colors)):\n            g=summary.loc[summary.method.eq(method)].set_index(\'dataset\').reindex(datasets)\n            y=g[key+\'_median\'].to_numpy(float);positive|=bool(np.isfinite(y).any())\n            if np.isfinite(y).any():\n                lo=g[key+\'_p25\'].to_numpy(float);hi=g[key+\'_p75\'].to_numpy(float)\n                ax.errorbar(np.arange(len(datasets))+(mi-3.5)*.015,y,yerr=np.vstack([y-lo,hi-y]),\n                    marker=\'o\',linestyle=\'--\' if method in STOCHASTIC else \'-\',markersize=4,capsize=2,color=color,label=method)\n        if positive:ax.set_yscale(\'log\');ax.legend(fontsize=11,ncol=4)\n        else:ax.text(.5,.5,\'No valid completed timing runs\',transform=ax.transAxes,ha=\'center\')\n        ax.set_xticks(np.arange(len(datasets)),[labels[x] for x in datasets],rotation=30,ha=\'right\')\n        ax.set_ylabel(\'Time (seconds; median and IQR, log scale)\' if positive else \'Time (seconds)\');ax.set_xlabel(\'Categorical benchmark dataset\')\n        ax.set_title(title+\' in the controlled CPU environment\');fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:\n            path=out/\'GAMBAR\'/(number+\'_\'+key+\'.\'+ext);path.parent.mkdir(parents=True,exist_ok=True);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n        plt.close(fig)\n    return paths\n\n\ndef archive_four_method_table(u,files,out,control):\n    old=out/\'TABEL/07_runtime_deterministik.csv\'\n    if old.is_file():\n        destination=control/\'ARSIP_VERSI_4_METODE\'/(datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\')+\'_\'+uuid.uuid4().hex[:8])\n        source=old.resolve();target=(destination/old.name).resolve()\n        if not source.is_relative_to(out.resolve()) or not target.is_relative_to(control.resolve()):raise ValueError(\'Lokasi arsip hasil di luar folder tahap31.\')\n        wanted=u.sha(source.read_bytes());destination.mkdir(parents=True,exist_ok=True);source.rename(target)\n        u.save_json(destination/\'asal.json\',{\'previous_path\':str(source.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n            \'sha256\':wanted,\'reason\':\'replaced by TABEL/07_runtime_8_metode.csv; former four-method result preserved\'})\n\n\ndef compatible_cache_migration(u,files,out,control,campaign_control,data,code,cfg,hw,source,utils_source):\n    """Adopt verified identical fits from the pinned eight-method I/O predecessor."""\n    if cfg[\'force_timing\']:\n        return {\'status\':\'disabled_by_force_timing\',\'adopted_timed_now\':0}\n    prior=PREVIOUS_EIGHT_METHOD_CODE\n    def definitions(text):\n        result={}\n        for node in ast.parse(text).body:\n            if isinstance(node,ast.FunctionDef):result[node.name]=u.digest(ast.dump(node))\n            elif isinstance(node,ast.ClassDef):\n                for method in node.body:\n                    if isinstance(method,ast.FunctionDef):result[node.name+\'.\'+method.name]=u.digest(ast.dump(method))\n        return result\n    constants={key:sorted(globals()[key]) if isinstance(globals()[key],set) else globals()[key] for key in prior[\'scientific_constants\']}\n    if constants!=prior[\'scientific_constants\']:\n        return {\'status\':\'scientific_constants_differ\',\'adopted_timed_now\':0}\n    for actual,expected in [(definitions(source),prior[\'scientific_definitions\']),\n                            (definitions(utils_source),prior[\'unchanged_utils_definitions\'])]:\n        if any(actual.get(key)!=value for key,value in expected.items()):\n            return {\'status\':\'scientific_or_cache_functions_differ\',\'adopted_timed_now\':0}\n    previous_code={**code,**prior[\'code\']}\n    if any(previous_code[key]!=code[key] for key in [\'helper25_sha256\',\'bk25_sha256\']):\n        return {\'status\':\'initializer_sources_differ\',\'adopted_timed_now\':0}\n    previous_id=u.digest({\'code\':previous_code,\'hardware\':hw,\'configuration\':timing_config(cfg),\'sources\':data[\'watched\']})[:20]\n    previous=control/\'CAMPAGNES\'/previous_id\n    expected={\'code\':previous_code,\'hardware\':hw,\'configuration\':timing_config(cfg),\'campaign\':previous_id}\n    if previous==campaign_control or u.read_checkpoint(previous/\'campaign.json\')!=expected:\n        return {\'status\':\'no_identical_predecessor_on_same_VM\',\'adopted_timed_now\':0}\n    counts={\'TIMED\':0,\'WARMUP\':0};rejected=[]\n    full=data[\'source\'].loc[data[\'source\'].scope.eq(\'BENCHMARK_FULL\')]\n    for folder,phase in [(\'ATTEMPTS\',\'TIMED\'),(\'WARMUP\',\'WARMUP\')]:\n        for oldpath in sorted((previous/folder).rglob(\'*.json\')):\n            destination=campaign_control/folder/oldpath.relative_to(previous/folder)\n            if destination.exists():continue\n            try:\n                raw=u.read_checkpoint(oldpath);row=raw[\'row\'];ds=row[\'dataset\'];method=row[\'method\'];block=int(row[\'block\'])\n                if ds not in cfg[\'benchmark_datasets\'] or method not in TIMING_METHODS or row[\'phase\']!=phase:\n                    raise ValueError(\'attempt identity\')\n                limit=cfg[\'warmup_repetitions\'] if phase==\'WARMUP\' else cfg[\'timing_blocks\']\n                if not 0<=block<limit:raise ValueError(\'attempt block\')\n                reference=full.loc[full.dataset.eq(ds)&full.method.eq(method)&full.run.eq(seed_for(method,block)[1])]\n                if len(reference)!=1:raise ValueError(\'source reference\')\n                ref=reference.iloc[0];ctx={\'dataset\':ds,\'k\':int(ref.k),\'matrix_sha256\':ref.matrix_sha256}\n                science=scientific_attempt(ctx,method,block,data,phase==\'WARMUP\')\n                identity={\'dataset\':ds,\'method\':method,\'block\':block,\'phase\':phase,\'hardware_sha256\':hw[\'identity_sha256\'],\n                    \'matrix_sha256\':ref.matrix_sha256,\'seed\':science[\'seed\'],\'source_run\':science[\'source_run\'],\n                    \'method_type\':\'stochastic\' if method in STOCHASTIC else \'deterministic\',\n                    \'k\':int(ref.k),\'n\':int(ref.sample_n),\'d\':int(ref.features)}\n                if any(row.get(key)!=value for key,value in identity.items()) or raw.get(\'scientific\')!=science:\n                    raise ValueError(\'matrix/seed/identity\')\n                old_signature=u.digest({\'code\':previous_code,\'hardware\':hw,\'scientific\':science,\'configuration\':timing_config(cfg)})\n                record=u.cache_read(files,oldpath,old_signature)\n                if record is None:raise ValueError(\'checksum/signature/output content\')\n                new_signature=u.digest({\'code\':code,\'hardware\':hw,\'scientific\':science,\'configuration\':timing_config(cfg)})\n                u.cache_write(files,destination,new_signature,[files.path(p) for p in record[\'outputs\']],{\n                    \'row\':record[\'row\'],\'scientific\':science,\'cache_migration\':{\n                        \'from_campaign\':previous_id,\'from_record\':str(oldpath.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n                        \'previous_record_sha256\':raw[\'record_sha256\'],\n                        \'reason\':\'verified identical scientific/timing functions; notebook autosave I/O repair only\'}})\n                counts[phase]+=1\n            except (OSError,RuntimeError,ValueError,KeyError,TypeError,AttributeError) as error:\n                rejected.append({\'record\':str(oldpath.relative_to(files.root)).replace(\'\\\\\',\'/\'),\'reason\':str(error)})\n    report_path=campaign_control/\'COMPATIBILITY_MIGRATION\'/(previous_id+\'.json\');old_report=u.read_checkpoint(report_path)\n    report={\'status\':\'verified_compatible_predecessor\',\'from_campaign\':previous_id,\'to_campaign\':campaign_control.name,\n        \'adopted_timed_now\':counts[\'TIMED\'],\'adopted_warmups_now\':counts[\'WARMUP\'],\n        \'adopted_timed_total\':old_report.get(\'adopted_timed_total\',0)+counts[\'TIMED\'],\n        \'adopted_warmups_total\':old_report.get(\'adopted_warmups_total\',0)+counts[\'WARMUP\'],\n        \'scientific_functions_verified\':True,\'same_sources_configuration_runtime_hardware_and_VM\':True,\n        \'failed_completed_attempts\':\'adopted under the same validation rules; never omitted or silently retried\',\n        \'original_records\':\'kept unchanged in from_campaign, including earlier interruptions/replaced records\',\n        \'rejected_records\':rejected,\'checked_utc\':utc()}\n    u.save_json(report_path,report)\n    print(\'Cache versi sebelum perbaikan autosave:\',counts[\'TIMED\'],\'timing dan\',counts[\'WARMUP\'],\'warm-up tervalidasi; ditolak:\',len(rejected),flush=True)\n    return report\n\n\ndef run(root,configuration,source,helper_source,bk_source,utils_source,u,h,bk):\n    from threadpoolctl import threadpool_limits\n    files=u.Files(Path(root).expanduser().resolve());out=files.root/\'OUTPUT/TAMBAHAN\'/STAGE;control=out/\'KONTROL\'\n    out.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    counts={\'MEASURE\':0,\'CACHE\':0};rows=[];new_warmups=[];run_id=datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\')+\'_\'+uuid.uuid4().hex[:8]\n    try:\n        data=source_setup(u,h,files,helper_source,bk_source);cfg=config_check(configuration,data[\'gd\'].DATASET_ORDER)\n        own=u.notebook_hash(files,STAGE,\'runtime31_configuration\',\'KONFIGURASI_31\')\n        code={\'implementation_sha256\':u.sha(source.encode()),\'helper25_sha256\':u.sha(helper_source.encode()),\'bk25_sha256\':u.sha(bk_source.encode()),\n            \'utils_sha256\':u.sha(utils_source.encode()),\'notebook_code\':own,\'runtime\':u.runtime(),\'libraries\':data[\'protocol\'][\'code\'][\'project_libraries\']}\n        checks=data[\'checks\']+[(STAGE,\'runtime31_configuration\',\'KONFIGURASI_31\',own)]\n        u.unchanged(files,data[\'watched\'],checks)\n        u.save_json(out/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code,\'revision\':\'eight-method controlled runtime\',\n            \'operational_fix\':\'bounded stable-JSON notebook reads during Colab autosave; code guards remain active\',\n            \'methods_quality\':h.METHODS,\'methods_timing\':TIMING_METHODS,\'deterministic_methods\':DETERMINISTIC,\'stochastic_methods\':sorted(STOCHASTIC),\n            \'seed_policy\':SEED_POLICY,\'stochastic_source_seeds\':SOURCE_SEEDS,\'seeds_per_stochastic_method\':10,\n            \'planned_timed_attempts\':len(cfg[\'benchmark_datasets\'])*len(TIMING_METHODS)*cfg[\'timing_blocks\'],\n            \'initial_warmup_plan\':len(cfg[\'benchmark_datasets\'])*len(TIMING_METHODS)*cfg[\'warmup_repetitions\'],\n            \'k\':\'dataset reference-class count, from unchanged registry; no class labels passed to initializer/optimizer\',\n            \'CARE\':data[\'constants\'][\'FROZEN_CARE_CONFIG\'],\'max_iter\':100,\'measurement\':\'fresh full-dataset initializer plus common K-Modes, median/IQR and mean/SD\',\n            \'excluded_from_clock\':\'loading/discretization, hash/validation, DBI/SI/ARI, output I/O and plotting\',\n            \'reuse\':\'all verified source25 quality and ten-dataset statistics; old source timing is not comparative timing\',\n            \'failure_denominators\':\'historical full440, resampling2400, new timed attempts, warmups and interruptions kept separate\',\n            \'BK_policy\':bk.IMPLEMENTATION_POLICY,\'selection\':\'none; frozen matrix, k/q, optimizer and source seeds; no best seed/run selection\'})\n        u.save_csv(out/\'PROTOKOL/06_rencana_seed_dan_urutan.csv\',pd.DataFrame([\n            {\'block\':block,\'order_position\':pos,\'method\':method,\'method_type\':\'stochastic\' if method in STOCHASTIC else \'deterministic\',\n             \'seed\':seed_for(method,block)[0],\'source25_run\':seed_for(method,block)[1]}\n            for block in range(cfg[\'timing_blocks\'])\n            for pos,method in enumerate(TIMING_METHODS[block%len(TIMING_METHODS):]+TIMING_METHODS[:block%len(TIMING_METHODS)])]))\n        data=audit_source(u,h,files,out,control,data,code,cfg)\n        benchmark_preflight(u,files,data,cfg,out)\n        u.unchanged(files,data[\'watched\'],checks)\n        light={p:v for p,v in data[\'watched\'].items() if p.startswith((\'DATA/\',\'LIB/\',U25+\'/PROTOKOL/\')) or p in [U25+\'/01_hasil_per_run_8_metode.csv\',U25+\'/02_ringkasan_8_metode.csv\']}\n        with threadpool_limits(limits=1):\n            hw=hardware(u);campaign=u.digest({\'code\':code,\'hardware\':hw,\'configuration\':timing_config(cfg),\'sources\':data[\'watched\']})[:20]\n            campaign_control=control/\'CAMPAGNES\'/campaign;campaign_control.mkdir(parents=True,exist_ok=True)\n            u.save_json(out/\'PROTOKOL/02_lingkungan_runtime.json\',hw)\n            u.save_json(campaign_control/\'campaign.json\',{\'code\':code,\'hardware\':hw,\'configuration\':timing_config(cfg),\'campaign\':campaign})\n            migration=compatible_cache_migration(u,files,out,control,campaign_control,data,code,cfg,hw,source,utils_source)\n            hc=dict(data[\'protocol\'][\'config\']);hc.update(scopes=[\'BENCHMARK_FULL\'],benchmark_datasets=cfg[\'benchmark_datasets\'])\n            for ctx in h.contexts(files,hc,data[\'gd\'],data[\'constants\'],{}):\n                u.unchanged(files,light,checks);ds=ctx[\'dataset\']\n                full=data[\'source\'].loc[data[\'source\'].scope.eq(\'BENCHMARK_FULL\')&data[\'source\'].dataset.eq(ds)]\n                if set(full.matrix_sha256)!=set([ctx[\'matrix_sha256\']]):raise RuntimeError(\'Matriks benchmark hasil loader berbeda dari sumber25: \'+ds)\n                ctx[\'reference\']={}\n                for method in TIMING_METHODS:\n                    group=full.loc[full.method.eq(method)]\n                    expected_runs=list(range(10)) if method in STOCHASTIC else [0]\n                    if sorted(group.run.astype(int))!=expected_runs:raise RuntimeError(\'Bank seed sumber tidak lengkap: \'+ds+\'/\'+method)\n                    ctx[\'reference\'][method]={}\n                    for r in group.to_dict(\'records\'):\n                        index=int(r[\'run\'])\n                        if int(r[\'seed\'])!=SOURCE_SEEDS[index]:raise RuntimeError(\'Seed sumber25 berbeda dari protokol runtime.\')\n                        ctx[\'reference\'][method][index]=r\n                u.save_json(out/\'PROTOKOL/MATRIKS\'/(ds+\'.json\'),{\'dataset\':ds,\'n\':len(ctx[\'X\']),\'d\':ctx[\'X\'].shape[1],\'k\':ctx[\'k\'],\n                    \'features\':ctx[\'features\'],\'matrix_sha256\':ctx[\'matrix_sha256\'],\'sources\':{p:files.sha(p) for p in ctx[\'sources\']},\'source25_matrix_verified\':True})\n                print(\'Dataset\',ds,\'n=\',len(ctx[\'X\']),\'d=\',ctx[\'X\'].shape[1],\'k=\',ctx[\'k\'],flush=True)\n                rr,ww,cc=timing_dataset(u,h,files,out,campaign_control,ctx,data,code,cfg,hw,bk,run_id)\n                rows.extend(rr);new_warmups.extend(ww)\n                for k,v in cc.items():counts[k]+=v\n                u.save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_timed_attempts\':len(rows),\'timing_counts\':counts,\'last_dataset\':ds,\'campaign\':campaign})\n        raw=pd.DataFrame(rows);planned=len(cfg[\'benchmark_datasets\'])*len(TIMING_METHODS)*cfg[\'timing_blocks\']\n        if len(raw)!=planned or raw.groupby([\'dataset\',\'block\']).method.nunique().ne(len(TIMING_METHODS)).any():raise RuntimeError(\'Jumlah timed attempts tidak lengkap.\')\n        summary=runtime_summary(raw);seed_summary=runtime_summary(raw,True);failure=failure_summary(raw)\n        warmup_rows=[]\n        for path in (campaign_control/\'WARMUP\').rglob(\'*.json\'):\n            rec=u.read_checkpoint(path)\n            if rec.get(\'status\')==\'selesai\' and \'row\' in rec:warmup_rows.append(rec[\'row\'])\n        interruptions=[]\n        for path in (campaign_control/\'INTERRUPTIONS\').glob(\'*.json\'):\n            rec=u.read_checkpoint(path);interruptions.append({\'file\':str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n                \'status\':rec.get(\'status\'),\'signature\':rec.get(\'signature\'),**rec.get(\'identity\',{})})\n        replaced=[]\n        for path in (campaign_control/\'REPLACED_ATTEMPTS\').glob(\'*.json\'):\n            archive=u.read_checkpoint(path);prior=archive.get(\'previous_record\',{}).get(\'row\',{})\n            replaced.append({\'archive\':str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\'reason\':archive.get(\'reason\'),\n                \'previous_record_checksum_valid\':archive.get(\'previous_record_checksum_valid\'),**prior})\n        for name,frame in [(\'RUNTIME/01_per_attempt.csv\',raw),(\'RUNTIME/02_ringkasan.csv\',summary),(\'RUNTIME/03_warmup.csv\',pd.DataFrame(warmup_rows)),\n            (\'RUNTIME/04_ringkasan_per_seed.csv\',seed_summary),\n            (\'AUDIT_RUNTIME/01_per_attempt.csv\',raw),(\'AUDIT_RUNTIME/02_per_dataset_metode.csv\',failure),\n            (\'AUDIT_RUNTIME/04_pengukuran_sebelumnya.csv\',pd.DataFrame(replaced,columns=[\'archive\',\'reason\',\'previous_record_checksum_valid\']+list(raw.columns))),\n            (\'AUDIT_RUNTIME/03_interupsi_sesi.csv\',pd.DataFrame(interruptions,columns=[\'file\',\'status\',\'signature\',\'dataset\',\'method\',\'block\',\'order_position\',\'phase\',\'hardware_sha256\',\'matrix_sha256\',\'run_id\'])),\n            (\'TABEL/07_runtime_8_metode.csv\',summary),(\'TABEL/08_failure_runtime.csv\',failure),(\'TABEL/09_runtime_per_seed.csv\',seed_summary)]:u.save_csv(out/name,frame)\n        archive_four_method_table(u,files,out,control)\n        full=cfg[\'benchmark_datasets\']==data[\'gd\'].DATASET_ORDER and cfg[\'timing_blocks\']>=30 and cfg[\'timing_blocks\']%len(SOURCE_SEEDS)==0 and cfg[\'warmup_repetitions\']>=1\n        nfail=int(raw.outcome.ne(\'OK\').sum());mismatches=int(raw.reference_match.eq(False).sum());inconsistent=int(summary.within_seed_repeats_consistent.eq(False).sum())\n        state=\'selesai_dengan_kegagalan_algoritmik\' if nfail else \'selesai_dengan_perbedaan_reproduksi\' if mismatches or inconsistent else \'selesai\'\n        u.save_json(out/\'PROTOKOL/03_denominator_dan_interpretasi.json\',{\'historical_full\':440,\'historical_resampling\':2400,\'historical_failure_runs\':int(data[\'historical\'].algorithmic_failure.sum()),\n            \'new_planned_timed_attempts\':planned,\'new_complete_timed_attempts\':len(raw),\'new_failed_timed_attempts\':nfail,\n            \'warmup_attempts_this_execution\':len(new_warmups),\'warmup_attempts_campaign\':len(warmup_rows),\'interrupted_attempt_records\':len(interruptions),\n            \'replaced_attempt_records\':len(replaced),\'replaced_policy\':\'previous forced/invalidated measurements preserved separately; failures are not silently removed\',\n            \'independence\':\'repeated full-data timings and overlapping resamples are not independent datasets\',\n            \'cap\':\'iteration cap flagged separately; cap reached alone is not proof of nonconvergence\',\n            \'CV\':\'CV>1 is size-imbalance diagnostic, not automatic failure\',\'failure_times\':\'failed attempts retained; successful timing summaries exclude failed fits\',\n            \'source_comparison\':\'raw quality is reused; new replay mismatches reported, never forced equal\',\n            \'contribution\':\'algorithmic benchmark only, not FSLSM/ILS transportability\',\n            \'seed_policy\':SEED_POLICY,\'stochastic_repetitions\':\'default10 source seeds x3 timing repetitions; each seed compared to its own source25 run\',\n            \'consistency\':\'deterministic across all repeats; stochastic within the same seed only, not across different seeds\',\n            \'table_scope\':\'eight-method quality/statistics supports updated manuscript Tables4,7,8–10; all eight methods timed in one campaign\'})\n        figsig=u.digest({\'code\':code,\'summary\':u.sha((out/\'RUNTIME/02_ringkasan.csv\').read_bytes())})\n        if not u.cache_read(files,control/\'gambar.json\',figsig,cfg[\'force_audit\']):u.cache_write(files,control/\'gambar.json\',figsig,figures(u,out,summary,cfg[\'benchmark_datasets\']),{})\n        u.unchanged(files,data[\'watched\'],checks);u.save_json(out/\'PROTOKOL/05_manifest_masukan.json\',data[\'watched\'])\n        status={\'status\':state,\'full_protocol\':full,\'historical_rows_verified\':3272,\'historical_benchmark_runs\':2840,\'BK_benchmark_runs_reused\':310,\n            \'historical_benchmark_failures\':int(data[\'historical\'].algorithmic_failure.sum()),\'historical_audit_cached\':data[\'historical_cached\'],\n            \'timing_counts\':counts,\'new_planned_timed_attempts\':planned,\'complete_timed_attempts\':len(raw),\'timed_failures\':nfail,\n            \'reference_mismatches\':mismatches,\'inconsistent_within_seed_groups\':inconsistent,\n            \'inconsistent_deterministic_groups\':int(summary.loc[summary.method_type.eq(\'deterministic\'),\'within_seed_repeats_consistent\'].eq(False).sum()),\n            \'warmups_this_execution\':len(new_warmups),\n            \'warmups_campaign\':len(warmup_rows),\'interruption_records\':len(interruptions),\'new_quality_fits\':0,\'new_runtime_fits\':counts[\'MEASURE\'],\n            \'replaced_attempt_records\':len(replaced),\n            \'timing_methods\':TIMING_METHODS,\'n_timing_methods\':len(TIMING_METHODS),\'seed_policy\':SEED_POLICY,\n            \'cache_migration\':migration,\n            \'n_datasets\':len(cfg[\'benchmark_datasets\']),\'campaign\':campaign,\'hardware_sha256\':hw[\'identity_sha256\'],\n            \'output\':str(out),\'finished_utc\':utc()}\n        manifest={str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):u.sha(p.read_bytes()) for p in out.rglob(\'*\') if p.is_file() and control not in p.parents}\n        u.save_json(control/\'manifest_keluaran.json\',manifest);u.save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap31 selesai:\',status,flush=True)\n        return {\'runtime\':summary,\'runtime_per_seed\':seed_summary,\'failure\':failure,\'historical\':summarize_history(data[\'historical\'],False),\'status\':status,\'output\':str(out)}\n    except BaseException as error:\n        completed_in_campaign=0\n        if \'campaign_control\' in locals():\n            completed_in_campaign=sum(u.read_checkpoint(p).get(\'status\')==\'selesai\' for p in (campaign_control/\'ATTEMPTS\').rglob(\'*.json\'))\n        u.save_json(control/\'status_eksekusi.json\',{\'status\':\'masukan_belum_lengkap\' if isinstance(error,FileNotFoundError) else \'terhenti\',\n            \'complete_dataset_attempts_this_execution\':len(rows),\'completed_attempt_records_in_campaign\':completed_in_campaign,\n            \'timing_counts\':counts,\'error\':type(error).__name__+\': \'+str(error),\n            \'resume\':\'jalankan notebook31 yang sama; completed attempt cache diverifikasi; VM berbeda memicu campaign timing baru\',\'finished_utc\':utc()})\n        raise\n'

_runtime31 = types.ModuleType('_runtime31')
exec(compile(RUNTIME31_SOURCE, '_runtime31', 'exec'), _runtime31.__dict__)


## Audit dan cara menafsirkan

Audit historis 2.840 benchmark memakai delapan metode, termasuk BK-Modes.
Attempt timing baru memakai seluruh delapan metode. Warm-up dan
interupsi sesi dicatat terpisah; keduanya tidak disamarkan sebagai run
timing sukses. Exception disimpan per attempt, tanpa mengganti seed atau
menghilangkan kegagalan dari denominator.

Catat empty final cluster, objective meningkat, angka tidak valid, iterasi
nonpositif, exception/resource error, dan cap. Cap=100 sendiri belum
membuktikan tidak konvergen tepat di iterasi terakhir. Cluster CV>1 merupakan
diagnosis ketimpangan, bukan otomatis failure. Final cluster yang tidak
kosong tidak membuktikan tidak pernah ada empty repair internal.

Fingerprint mode, label, cost dan iterasi dipakai untuk memeriksa konsistensi
deterministik dan pengulangan dalam seed stokastik yang sama. Variasi antar-seed
tidak dianggap ketidakstabilan deterministik. Perbedaan terhadap sumber dilaporkan, tanpa memaksakan angka
agar sama. Metrik kualitas sumber tidak ditimpa. Iterasi akhir lebih sedikit
tidak otomatis berarti total runtime lebih kecil, terutama jika ada biseksi
internal pada initializer.

BK-Modes tetap implementasi ulang Python dari arXiv:2502.07081v2,
Algorithms 8–9 (15 Februari 2025), dengan kebijakan pada IMPLEMENTATION_POLICY.
Versi sumber adalah preprint, bukan kode C resmi penulis. Runtime di sini
bersyarat pada implementasi dan mesin yang dicatat.


In [10]:
HASIL_31 = _runtime31.run(ROOT_31, KONFIGURASI_31, RUNTIME31_SOURCE,
                        HELPER25_SOURCE, BK25_SOURCE, UTILS31_SOURCE,
                        _u31, _h25, _bk25)
display(HASIL_31['runtime'])
display(HASIL_31['runtime_per_seed'])
display(HASIL_31['failure'])
display(HASIL_31['historical'])
print(HASIL_31['status'])
print('Folder hasil:', HASIL_31['output'])


Audit sumber: 500 / 3272
Audit sumber: 1000 / 3272
Audit sumber: 1500 / 3272
Audit sumber: 2000 / 3272
Audit sumber: 2500 / 3272
Audit sumber: 3000 / 3272
Kualitas/audit:3272 catatan terverifikasi;2840 benchmark,310 BK-Modes.
Cache versi sebelum perbaikan autosave: 1440 timing dan 48 warm-up tervalidasi; ditolak: 0
Dataset car n= 1728 d= 6 k= 4
Runtime car blok 1 / 30 {'MEASURE': 0, 'CACHE': 8}
Runtime car blok 2 / 30 {'MEASURE': 0, 'CACHE': 16}
Runtime car blok 3 / 30 {'MEASURE': 0, 'CACHE': 24}
Runtime car blok 4 / 30 {'MEASURE': 0, 'CACHE': 32}
Runtime car blok 5 / 30 {'MEASURE': 0, 'CACHE': 40}
Runtime car blok 6 / 30 {'MEASURE': 0, 'CACHE': 48}
Runtime car blok 7 / 30 {'MEASURE': 0, 'CACHE': 56}
Runtime car blok 8 / 30 {'MEASURE': 0, 'CACHE': 64}
Runtime car blok 9 / 30 {'MEASURE': 0, 'CACHE': 72}
Runtime car blok 10 / 30 {'MEASURE': 0, 'CACHE': 80}
Runtime car blok 11 / 30 {'MEASURE': 0, 'CACHE': 88}
Runtime car blok 12 / 30 {'MEASURE': 0, 'CACHE': 96}
Runtime car blok 13 / 30 {'

,dataset,method,n,d,k,n_attempted,n_success,n_failure,failure_rate,n_reference_mismatch,...,method_type,n_unique_seeds,source_runs,within_seed_repeats_consistent,deterministic_repeats_consistent,n_seed_groups_with_repeats,n_distinct_initial_mode_sets,init_sec_ratio_to_Cao,kmodes_sec_ratio_to_Cao,total_sec_ratio_to_Cao
0,car,Cao,1728,6,4,30,30,0,0.0,0,...,deterministic,1,[0],True,True,1,1,1.000000,1.000000,1.000000
1,car,Bai,1728,6,4,30,30,0,0.0,0,...,deterministic,1,[0],True,True,1,1,80.006238,0.942462,29.903653
2,car,CARE-KM,1728,6,4,30,30,0,0.0,0,...,deterministic,1,[0],True,True,1,1,1.991811,0.973756,1.358985
3,car,BK-Modes,1728,6,4,30,30,0,0.0,0,...,deterministic,1,[0],True,True,1,1,6.467827,0.931003,2.964742
4,car,Random,1728,6,4,30,30,0,0.0,0,...,stochastic,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",True,None,10,10,0.041524,1.010899,0.666401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,nursery,BK-Modes,12960,8,5,30,30,0,0.0,0,...,deterministic,1,[0],True,True,1,1,8.209681,0.986535,3.672649
76,nursery,Random,12960,8,5,30,30,0,0.0,0,...,stochastic,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",True,None,10,10,0.004393,0.986619,0.615572
77,nursery,Huang,12960,8,5,30,30,0,0.0,0,...,stochastic,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",True,None,10,10,0.873557,0.993494,0.948737
78,nursery,K-Modes++,12960,8,5,30,30,0,0.0,0,...,stochastic,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",True,None,10,10,0.166506,0.993773,0.686107


,dataset,method,n,d,k,n_attempted,n_success,n_failure,failure_rate,n_reference_mismatch,...,source_runs,within_seed_repeats_consistent,deterministic_repeats_consistent,n_seed_groups_with_repeats,n_distinct_initial_mode_sets,seed,source_run,init_sec_ratio_to_Cao,kmodes_sec_ratio_to_Cao,total_sec_ratio_to_Cao
0,car,Cao,1728,6,4,30,30,0,0.0,0,...,[0],True,True,1,1,120000,0,1.000000,1.000000,1.000000
1,car,Bai,1728,6,4,30,30,0,0.0,0,...,[0],True,True,1,1,120000,0,80.006238,0.942462,29.903653
2,car,CARE-KM,1728,6,4,30,30,0,0.0,0,...,[0],True,True,1,1,120000,0,1.991811,0.973756,1.358985
3,car,BK-Modes,1728,6,4,30,30,0,0.0,0,...,[0],True,True,1,1,120000,0,6.467827,0.931003,2.964742
4,car,Random,1728,6,4,3,3,0,0.0,0,...,[0],True,None,1,1,120000,0,0.043041,0.970946,0.641300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,nursery,HR-2023,12960,8,5,3,3,0,0.0,0,...,[8],True,None,1,1,120008,8,0.898465,1.845149,1.484122
436,nursery,Random,12960,8,5,3,3,0,0.0,0,...,[9],True,None,1,1,120009,9,0.004172,0.962461,0.600334
437,nursery,Huang,12960,8,5,3,3,0,0.0,0,...,[9],True,None,1,1,120009,9,0.868472,1.004619,0.945468
438,nursery,K-Modes++,12960,8,5,3,3,0,0.0,0,...,[9],True,None,1,1,120009,9,0.257114,1.093778,0.776708


,dataset,method,method_type,n_unique_seeds,n_attempted,n_success,n_failed,empty_final_cluster_runs,objective_worsened_runs,nonpositive_iterations_runs,invalid_numeric_runs,iteration_cap_reached_runs,cluster_cv_gt_1_runs,resource_error_runs,initializer_error_runs,optimizer_error_runs,audit_error_runs,reference_mismatch_runs,failure_rate,elapsed_to_failure_sec_median
0,car,Cao,deterministic,1,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
1,car,Bai,deterministic,1,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
2,car,CARE-KM,deterministic,1,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
3,car,BK-Modes,deterministic,1,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
4,car,Random,stochastic,10,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,nursery,BK-Modes,deterministic,1,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
76,nursery,Random,stochastic,10,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
77,nursery,Huang,stochastic,10,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN
78,nursery,K-Modes++,stochastic,10,30,30,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN


,scope,method,n_runs,n_success,n_failures,failure_rate,invalid_numeric_runs,metric_undefined_runs,empty_final_cluster_runs,objective_worsened_runs,nonpositive_iterations_runs,iteration_cap_reached_runs,cluster_cv_gt_1_runs,source_invalid_runs,algorithmic_failure_runs
0,BENCHMARK_FULL,Random,100,100,0,0.0,0,0,0,0,0,0,0,0,0
1,BENCHMARK_FULL,Huang,100,100,0,0.0,0,0,0,0,0,0,0,0,0
2,BENCHMARK_FULL,Cao,10,10,0,0.0,0,0,0,0,0,0,1,0,0
3,BENCHMARK_FULL,Bai,10,10,0,0.0,0,0,0,0,0,0,0,0,0
4,BENCHMARK_FULL,K-Modes++,100,100,0,0.0,0,0,0,0,0,0,0,0,0
5,BENCHMARK_FULL,HR-2023,100,100,0,0.0,0,0,0,0,0,0,0,0,0
6,BENCHMARK_FULL,CARE-KM,10,10,0,0.0,0,0,0,0,0,0,1,0,0
7,BENCHMARK_FULL,BK-Modes,10,10,0,0.0,0,0,0,0,0,0,0,0,0
8,BENCHMARK_RESAMPLING,Random,300,300,0,0.0,0,0,0,0,0,0,0,0,0
9,BENCHMARK_RESAMPLING,Huang,300,300,0,0.0,0,0,0,0,0,0,0,0,0


{'status': 'selesai', 'full_protocol': True, 'historical_rows_verified': 3272, 'historical_benchmark_runs': 2840, 'BK_benchmark_runs_reused': 310, 'historical_benchmark_failures': 0, 'historical_audit_cached': False, 'timing_counts': {'MEASURE': 960, 'CACHE': 1440}, 'new_planned_timed_attempts': 2400, 'complete_timed_attempts': 2400, 'timed_failures': 0, 'reference_mismatches': 0, 'inconsistent_within_seed_groups': 0, 'inconsistent_deterministic_groups': 0, 'warmups_this_execution': 32, 'warmups_campaign': 80, 'interruption_records': 0, 'new_quality_fits': 0, 'new_runtime_fits': 960, 'replaced_attempt_records': 0, 'timing_methods': ['Cao', 'Bai', 'CARE-KM', 'BK-Modes', 'Random', 'Huang', 'K-Modes++', 'HR-2023'], 'n_timing_methods': 8, 'seed_policy': 'stochastic: source25 full-data seeds120000..120009, cycled every10 blocks; deterministic: seed120000', 'cache_migration': {'status': 'verified_compatible_predecessor', 'from_campaign': '456740512cd2f3b9bf1b', 'to_campaign': '48b7b4547304d1

## Keluaran

- `KUALITAS/`: hasil/ringkasan delapan metode, 55 verifikasi, dan statistik
  sepuluh dataset yang digunakan ulang dari 25, dengan sumber terverifikasi.
- `AUDIT_HISTORIS/`: status/jenis failure per run, dataset/metode, dan scope;
  full 440 serta resampling 2.400 tetap dibedakan.
- `RUNTIME/`: 2.400 attempt timing default, ringkasan pooled/per-seed, dan warm-up.
- `AUDIT_RUNTIME/`: status per attempt, ringkasan failure, mismatch/cap dan
  interupsi sesi. Elapsed-to-failure dipisahkan dari runtime sukses.
- `PARTISI/`: mode/assignment dan trace BK pada blok pertama tiap
  dataset/metode, untuk memeriksa eksekusi ilmiah tanpa bank duplikat 30 kali.
- `TABEL/01–05`: kualitas dan rank delapan metode, termasuk bahan Tabel 4,
  7, 8–10. `06`: failure historis. `07_runtime_8_metode.csv`: runtime delapan
  metode. `08`: failure runtime baru. `09_runtime_per_seed.csv`: detail seed.
  Tabel lama `07_runtime_deterministik.csv`, jika ada, diarsipkan ke KONTROL.
- `GAMBAR/`: grafik initialization dan total runtime, PNG/PDF, font 11 pt.
- `PROTOKOL/`, `KONTROL/`: konfigurasi, hardware, data fingerprint, manifest,
  denominator, checkpoint per attempt dan campaign.
- `PROTOKOL/06_rencana_seed_dan_urutan.csv`: urutan metode, seed dan run sumber
  setiap blok. Tabel per-seed membedakan variasi seed dari variasi pengukuran.

Unit statistik kualitas tetap sepuluh dataset, dengan mean 30 matched
resamples tiap dataset. Nol/ties, exact signed-rank dan Holm tujuh kontras
dibawa dari sumber. Repeated timing dan resample tumpang tindih tidak menjadi
dataset independen. Benchmark ini mendukung komponen clustering; penilaian
FSLSM/ILS serta transportabilitas tetap memiliki eksperimen tersendiri.
